# CS781 Phase 4: six court-judgment submissions

Attach the four **Phase 1 recovery** Kaggle Inputs (`phase1_submission01_recovery` through `phase1_submission04_recovery`), enable a **two-GPU** accelerator and Internet, and enable a Kaggle Secret named exactly **HF_TOKEN** with access to the official course Hugging Face dataset. Choose **Run All**. The notebook fetches `Phase_4_CJPE/Train/train.json`, `Train/dev.json`, and `Test/test.json` from one pinned dataset revision. It refuses an incomplete or changed schema. Phase 4 is scheduled to start on 17 October 2026 in the local specification; if the gated files are not released when you run, it stops with a clear error before training.

**Input-loader repair:** this build identifies each Phase 1 recovery by its actual model, tokenizer, and checkpoint files. It accepts an extracted Kaggle dataset or a ZIP even when Kaggle has changed its mount or archive name. If an input is truly absent, the error lists mounted inputs and matching files. Upload this revised notebook as a new Kaggle notebook so an old saved execution does not keep running the earlier loader.

**Repeated-case-ID repair:** the released `train.json` has repeated case IDs. Training and development rows now receive stable internal row keys, while the original case IDs remain available for grouping. Validation keeps related rows together where those IDs identify a case. Test submission IDs remain exactly as supplied in `test.json`.

The six methods are: (01) full fine-tuning of Phase 1 base 01, (02) partial layer-wise fine-tuning of base 03, (03) LORMA on base 04, (04) adaptive-rank AdaLoRA on base 02, (05) bottleneck adapters on base 03, and (06) distillation from the larger trained base-01 teacher to a base-03 student. All use the original Phase 1 weights. The classifier reads multiple chunks across each long case and predicts one binary outcome per case. No other pretrained model, corpus, or external LLM is loaded.

Each training stage uses both GPUs with one distributed process per GPU, mixed precision, gradient checkpointing, gradient accumulation, atomic checkpoints, and early stopping. The notebook first trains and packages one provisional submission per method. It then runs three **case-level outer folds** for each method. Checkpoints and thresholds are chosen on inner validation cases, while the outer cases remain untouched until scoring. It republishes each package with a threshold from the fold evidence and reports macro F1 for both classes. A weak local F1 is flagged rather than represented as a good private score.

Outputs: six `/kaggle/working/CS781_Phase4_Submission_XX.zip` files, each containing root-level `prediction.json`; and `/kaggle/working/CS781_Phase4_All_Artifacts.zip`, containing official data, Phase 1 sources/weights, model states, optimizer checkpoints for active stages, predictions, histories, reports, and code. To resume in a new session, attach the newest all-artifacts ZIP as an additional Kaggle Input with the four original Phase 1 inputs and Run All again. The artifact is rebuilt **after** distributed workers exit. A fatal Kaggle draft session can still make local files inaccessible; download or persist the recovery ZIP after a completed session.

The job may exceed one Kaggle session depending on the released dataset size and accelerator quota. After about 9.5 hours this notebook stops at a safe stage boundary with a recovery ZIP; the same notebook continues remaining stages on the next Run All. A strong private F1 cannot be guaranteed before Codabench scoring. Review the course's AI-assistance declaration requirements before using the results.

## Embedded recovery and Phase 1 model loader

In [ ]:
%%writefile /kaggle/working/cs781_phase4_utils.py
"""Shared recovery, Phase-1 loading, and LORMA helpers for Phase 4."""
from __future__ import annotations

import hashlib
import importlib.util
import json
import math
import os
import shutil
import sys
import tempfile
import zipfile
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

ROOT=Path('/kaggle/working/CS781_Capstone/phase4_cjpe')
INPUT=Path('/kaggle/input')
ARTIFACT=Path('/kaggle/working/CS781_Phase4_All_Artifacts.zip')
BASE_FILES={f'{i:02d}':f'phase1_submission{i:02d}_recovery.zip' for i in range(1,5)}
SOURCES={'01':'decoder_lm.py','02':'encoder_decoder_lm.py',
         '03':'prefix_hybrid_lm.py','04':'efficient_causal_lm.py'}


def sha(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(4<<20),b''):
            digest.update(block)
    return digest.hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf8'))


def write_json(path,value):
    target=Path(path)
    target.parent.mkdir(parents=True,exist_ok=True)

    with tempfile.NamedTemporaryFile(
        'w',
        encoding='utf8',
        dir=target.parent,
        delete=False
    ) as handle:

        temporary=Path(handle.name)

        json.dump(
            value,
            handle,
            indent=2,
            ensure_ascii=False
        )

        handle.write('\n')
        handle.flush()
        os.fsync(handle.fileno())

    os.replace(
        temporary,
        target
    )


def save_torch(path,value):
    target=Path(path)
    target.parent.mkdir(parents=True,exist_ok=True)

    with tempfile.NamedTemporaryFile(
        dir=target.parent,
        delete=False
    ) as handle:
        temporary=Path(handle.name)

    try:
        torch.save(
            value,
            temporary
        )

        os.replace(
            temporary,
            target
        )

    finally:
        temporary.unlink(
            missing_ok=True
        )


def find_one(name,required=True):
    matches=sorted(
        {
            p.resolve()
            for p in INPUT.rglob(name)
            if p.is_file()
        }
    )

    if len(matches)>1 or (
        required and len(matches)!=1
    ):
        raise RuntimeError(
            f'Expected exactly one {name} under '
            f'{INPUT}; found {matches}'
        )

    return matches[0] if matches else None


def safe_name(name):
    path=Path(name)

    return (
        bool(name)
        and not path.is_absolute()
        and '..' not in path.parts
        and '\\' not in name
    )


# ============================================================
# EPOCH-PATCH RECOVERY
# Supports BOTH:
#   1) CS781_Phase4_Epoch*_Resume_Patch.zip
#   2) Kaggle-extracted dataset folders containing resume_info.json + stage/
# ============================================================

def _resume_patch_candidates():
    candidates=[]

    # ---------- ZIP patches ----------
    for patch in sorted(INPUT.rglob('CS781_Phase4_Epoch*_Resume_Patch.zip')):
        if not patch.is_file():
            continue
        with zipfile.ZipFile(patch) as archive:
            names=archive.namelist()
            if 'resume_info.json' not in names:
                continue
            info=json.loads(archive.read('resume_info.json'))
            epoch=int(info['epoch'])
            relative_stage=str(info['relative_stage_directory'])
            if not safe_name(relative_stage):
                raise RuntimeError(f'Unsafe resume stage in {patch}: {relative_stage}')
            candidates.append((epoch, 'zip', patch, info))

    # ---------- Extracted Kaggle dataset patches ----------
    for info_path in sorted(INPUT.rglob('resume_info.json')):
        if not info_path.is_file():
            continue
        patch_root=info_path.parent
        stage_root=patch_root/'stage'
        latest=stage_root/'latest.pt'
        if not latest.is_file():
            continue
        info=read_json(info_path)
        if 'epoch' not in info or 'relative_stage_directory' not in info:
            continue
        epoch=int(info['epoch'])
        relative_stage=str(info['relative_stage_directory'])
        if not safe_name(relative_stage):
            raise RuntimeError(f'Unsafe extracted resume stage in {info_path}: {relative_stage}')
        candidates.append((epoch, 'dir', patch_root, info))

    return candidates


def restore_epoch_patch():
    candidates=_resume_patch_candidates()

    if not candidates:
        print('No epoch resume patch found.', flush=True)
        return None

    candidates.sort(
        key=lambda item:(
            item[0],
            item[2].stat().st_mtime
        )
    )

    saved_epoch,kind,source,info=candidates[-1]
    relative_stage=Path(info['relative_stage_directory'])
    stage_dir=ROOT/relative_stage
    stage_dir.mkdir(parents=True,exist_ok=True)

    print('Found epoch resume patch:', source, flush=True)
    print('Patch type:', kind, flush=True)
    print('Patch target stage:', relative_stage.as_posix(), flush=True)

    if kind=='zip':
        with zipfile.ZipFile(source) as archive:
            names=archive.namelist()
            if len(names)!=len(set(names)):
                raise RuntimeError(f'Duplicate members in resume patch: {source}')
            for member in ('stage/latest.pt','stage/best.pt','stage/history.json'):
                if member not in names:
                    continue
                target=stage_dir/Path(member).name
                with archive.open(member) as reader, tempfile.NamedTemporaryFile(dir=stage_dir,delete=False) as handle:
                    temporary=Path(handle.name)
                    shutil.copyfileobj(reader,handle)
                    handle.flush()
                    os.fsync(handle.fileno())
                os.replace(temporary,target)
                print('Restored patch file:', target, flush=True)
    else:
        src_stage=source/'stage'
        for filename in ('latest.pt','best.pt','history.json'):
            src=src_stage/filename
            if not src.is_file():
                continue
            target=stage_dir/filename
            with src.open('rb') as reader, tempfile.NamedTemporaryFile(dir=stage_dir,delete=False) as handle:
                temporary=Path(handle.name)
                shutil.copyfileobj(reader,handle)
                handle.flush()
                os.fsync(handle.fileno())
            os.replace(temporary,target)
            print('Restored patch file:', target, flush=True)

    latest=stage_dir/'latest.pt'
    if not latest.is_file():
        raise RuntimeError(f'Resume patch {source} did not restore latest.pt')

    try:
        state=torch.load(latest,map_location='cpu',weights_only=False)
    except TypeError:
        state=torch.load(latest,map_location='cpu')

    actual_epoch=int(state['epoch'])
    if actual_epoch!=saved_epoch:
        raise RuntimeError(
            f'Resume patch epoch mismatch: metadata={saved_epoch}, checkpoint={actual_epoch}'
        )

    print('Restored Phase-4 epoch patch:', source, flush=True)
    print('Resume stage:', relative_stage.as_posix(), flush=True)
    print(
        f'Checkpoint completed epoch {actual_epoch}; training will start from epoch {actual_epoch+1}.',
        flush=True
    )
    return actual_epoch


# ============================================================
# MAIN PHASE-4 ARTIFACT RECOVERY
# Supports BOTH:
#   1) CS781_Phase4_All_Artifacts.zip
#   2) Kaggle-extracted dataset folder containing integrity.json + run_config.json
# ============================================================

def _find_extracted_main_artifact():
    candidates=[]
    for integrity in sorted(INPUT.rglob('integrity.json')):
        root=integrity.parent
        if not (root/'run_config.json').is_file():
            continue
        # Strong signals that this is the Phase-4 artifact root.
        if not (root/'runs').is_dir():
            continue
        candidates.append(root)

    if not candidates:
        return None
    if len(candidates)>1:
        # Prefer a folder whose name looks like the intended dataset.
        preferred=[p for p in candidates if 'phase4' in p.name.lower() and 'artifact' in p.name.lower()]
        if len(preferred)==1:
            return preferred[0]
        raise RuntimeError(f'Multiple extracted Phase-4 artifact roots found: {candidates}')
    return candidates[0]


def _restore_extracted_artifact(source):
    manifest_path=source/'integrity.json'
    manifest=read_json(manifest_path)['files']

    print('Found extracted Phase-4 artifact:', source, flush=True)

    for relative,meta in manifest.items():
        if not safe_name(relative):
            raise RuntimeError(f'Unsafe recovery member {relative}')
        src=source/relative
        if not src.is_file():
            raise RuntimeError(f'Missing extracted recovery member: {src}')
        if src.stat().st_size!=int(meta['bytes']):
            raise RuntimeError(f'Size mismatch in extracted recovery member: {src}')
        if sha(src)!=meta['sha256']:
            raise RuntimeError(f'Hash mismatch in extracted recovery member: {src}')

        target=ROOT/relative
        target.parent.mkdir(parents=True,exist_ok=True)

        if target.exists():
            if sha(target)!=meta['sha256']:
                raise RuntimeError(f'Existing file conflicts with recovery: {relative}')
            continue

        with src.open('rb') as reader, tempfile.NamedTemporaryFile(dir=target.parent,delete=False) as handle:
            temporary=Path(handle.name)
            shutil.copyfileobj(reader,handle)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary,target)

    print('Restored extracted Phase-4 artifact:', source, flush=True)


def restore_artifact():
    # Restore main Phase-4 state only when working state is not already present.
    if not (ROOT/'run_config.json').is_file():
        zip_source=find_one(ARTIFACT.name,required=False)

        if zip_source is not None:
            with zipfile.ZipFile(zip_source) as archive:
                names=archive.namelist()
                if 'integrity.json' not in names or len(names)!=len(set(names)):
                    raise RuntimeError('Invalid Phase-4 recovery ZIP manifest')

                manifest=json.loads(archive.read('integrity.json'))['files']
                if set(manifest)!=set(names)-{'integrity.json'}:
                    raise RuntimeError('Phase-4 recovery member list differs from manifest')

                for name in names:
                    if name=='integrity.json':
                        continue
                    if not safe_name(name) or archive.getinfo(name).is_dir():
                        raise RuntimeError(f'Unsafe recovery member {name}')

                    target=ROOT/name
                    target.parent.mkdir(parents=True,exist_ok=True)

                    with archive.open(name) as reader, tempfile.NamedTemporaryFile(dir=target.parent,delete=False) as handle:
                        temporary=Path(handle.name)
                        digest=hashlib.sha256()
                        size=0
                        for block in iter(lambda:reader.read(4<<20),b''):
                            handle.write(block)
                            digest.update(block)
                            size+=len(block)
                        handle.flush()
                        os.fsync(handle.fileno())

                    if digest.hexdigest()!=manifest[name]['sha256'] or size!=manifest[name]['bytes']:
                        temporary.unlink(missing_ok=True)
                        raise RuntimeError(f'Corrupt recovery member {name}')

                    if target.exists():
                        if sha(target)!=digest.hexdigest():
                            temporary.unlink(missing_ok=True)
                            raise RuntimeError(f'Existing file conflicts with recovery: {name}')
                        temporary.unlink()
                    else:
                        os.replace(temporary,target)

            print('Restored Phase-4 artifact:',zip_source,flush=True)

        else:
            extracted=_find_extracted_main_artifact()
            if extracted is None:
                print('No Phase-4 all-artifacts ZIP or extracted artifact found; starting without main Phase-4 recovery.',flush=True)
            else:
                _restore_extracted_artifact(extracted)
    else:
        print('Existing Phase-4 working state found; main artifact restore skipped.',flush=True)

    # Always overlay the newest/highest epoch patch after main restore.
    restore_epoch_patch()


# ============================================================
# SNAPSHOT
# ============================================================

def snapshot():

    ROOT.mkdir(
        parents=True,
        exist_ok=True
    )

    files=sorted(
        p
        for p in ROOT.rglob('*')
        if (
            p.is_file()
            and
            'hf_cache'
            not in p.parts
        )
    )

    temporary=ARTIFACT.with_suffix(
        '.zip.tmp'
    )

    manifest={
        'version':1,
        'files':{}
    }

    with zipfile.ZipFile(
        temporary,
        'w',
        allowZip64=True
    ) as archive:

        for path in files:

            relative=(
                path
                .relative_to(ROOT)
                .as_posix()
            )

            if relative.endswith(
                '.tmp'
            ):
                continue

            manifest[
                'files'
            ][relative]={
                'sha256':sha(path),
                'bytes':path.stat().st_size
            }

            archive.write(
                path,
                relative,
                compress_type=(
                    zipfile.ZIP_STORED
                    if path.suffix=='.pt'
                    else
                    zipfile.ZIP_DEFLATED
                )
            )

        archive.writestr(
            'integrity.json',
            json.dumps(
                manifest,
                sort_keys=True
            )
        )

    os.replace(
        temporary,
        ARTIFACT
    )

    print(
        f'Updated {ARTIFACT} '
        f'({ARTIFACT.stat().st_size/2**30:.2f} GiB)',
        flush=True
    )


# ============================================================
# PHASE-1 BASE RECOVERY
# ============================================================

def stage_base(base_id):

    dest=(
        ROOT
        /
        'bases'
        /
        base_id
    )

    needed=[
        f'model/{SOURCES[base_id]}',
        'model/course_bpe.py',
        'checkpoints/selected_model.pt'
    ]

    if (
        all(
            (dest/name).is_file()
            for name in needed
        )
        and
        (
            dest
            /
            'tokenizer/tokenizer.json'
        ).is_file()
    ):

        return dest

    source=find_one(
        BASE_FILES[
            base_id
        ]
    )

    with zipfile.ZipFile(
        source
    ) as archive:

        names=archive.namelist()

        if (
            len(names)
            !=
            len(set(names))
            or
            any(
                not safe_name(name)
                for name in names
            )
        ):

            raise RuntimeError(
                f'Unsafe Phase-1 archive: '
                f'{source}'
            )

        wanted=(
            needed
            +
            [
                name
                for name in names
                if (
                    name.startswith(
                        'tokenizer/'
                    )
                    and
                    not name.endswith('/')
                )
            ]
        )

        if (
            not all(
                name in names
                for name in needed
            )
            or
            'tokenizer/tokenizer.json'
            not in wanted
        ):

            raise RuntimeError(
                f'Incomplete Phase-1 '
                f'archive: {source}'
            )

        for name in wanted:

            target=(
                dest
                /
                name
            )

            if target.is_file():
                continue

            target.parent.mkdir(
                parents=True,
                exist_ok=True
            )

            with (
                archive.open(name)
                as reader,
                tempfile.NamedTemporaryFile(
                    dir=target.parent,
                    delete=False
                ) as handle
            ):

                temporary=Path(
                    handle.name
                )

                shutil.copyfileobj(
                    reader,
                    handle
                )

            os.replace(
                temporary,
                target
            )

    write_json(
        dest
        /
        'provenance.json',
        {
            'archive_name':
                source.name,

            'archive_sha256':
                sha(source),

            'model_sha256':
                sha(
                    dest
                    /
                    needed[0]
                ),

            'checkpoint_sha256':
                sha(
                    dest
                    /
                    needed[2]
                )
        }
    )

    return dest


def source_module(path,name):

    spec=importlib.util.spec_from_file_location(
        name,
        path
    )

    module=importlib.util.module_from_spec(
        spec
    )

    sys.modules[
        name
    ]=module

    spec.loader.exec_module(
        module
    )

    return module


def load_base(base_id):

    directory=(
        ROOT
        /
        'bases'
        /
        base_id
    )

    model_path=(
        directory
        /
        'model'
        /
        SOURCES[
            base_id
        ]
    )

    source=source_module(
        model_path,
        f'phase4_model_'
        f'{base_id}_'
        f'{os.getpid()}'
    )

    tokenizer_source=source_module(
        directory
        /
        'model/course_bpe.py',

        f'phase4_bpe_'
        f'{base_id}_'
        f'{os.getpid()}'
    )

    tokenizer=(
        tokenizer_source
        .CourseBPETokenizer
        .from_pretrained(
            directory
            /
            'tokenizer'
        )
    )

    try:

        checkpoint=torch.load(
            directory
            /
            'checkpoints/selected_model.pt',

            map_location='cpu',
            weights_only=True
        )

    except TypeError:

        checkpoint=torch.load(
            directory
            /
            'checkpoints/selected_model.pt',

            map_location='cpu'
        )

    if (
        checkpoint.get(
            'model_source_sha256'
        )
        and
        checkpoint[
            'model_source_sha256'
        ]
        !=
        sha(model_path)
    ):

        raise RuntimeError(
            f'Phase-1 base '
            f'{base_id} '
            f'source hash mismatch'
        )

    metadata=read_json(
        directory
        /
        'tokenizer/'
        'tokenizer_metadata.json'
    )

    if (
        checkpoint.get(
            'tokenizer_fingerprint'
        )
        !=
        metadata.get(
            'fingerprint'
        )
    ):

        raise RuntimeError(
            f'Phase-1 base '
            f'{base_id} '
            f'tokenizer fingerprint mismatch'
        )

    if (
        int(
            checkpoint[
                'model_config'
            ][
                'vocab_size'
            ]
        )
        !=
        len(tokenizer)
    ):

        raise RuntimeError(
            f'Phase-1 base '
            f'{base_id} '
            f'vocabulary mismatch'
        )

    base=source.build_model(
        checkpoint[
            'model_config'
        ]
    )

    base.load_state_dict(
        checkpoint[
            'model_state'
        ],
        strict=True
    )

    return (
        base,
        tokenizer
    )


# ============================================================
# LORMA
# ============================================================

class LoRMA(nn.Module):

    def __init__(
        self,
        base,
        rank,
        alpha
    ):

        super().__init__()

        self.base=base

        self.a=nn.Linear(
            base.out_features,
            rank,
            bias=False
        )

        self.b=nn.Linear(
            rank,
            base.out_features,
            bias=False
        )

        nn.init.kaiming_uniform_(
            self.a.weight,
            a=math.sqrt(5)
        )

        nn.init.zeros_(
            self.b.weight
        )

        self.scale=(
            alpha
            /
            rank
        )

    def forward(
        self,
        x
    ):

        value=self.base(
            x
        )

        return (
            value
            +
            self.b(
                self.a(
                    F.linear(
                        x,
                        self.base.weight
                    )
                )
            )
            *
            self.scale
        )


## Embedded Phase 4 data, models, training, and submission code

In [ ]:
%%writefile /kaggle/working/cs781_phase4_cjpe_runner.py
"""CS781 Phase 4 CJPE: six resumable, two-GPU Phase-1 adaptations.

This source is embedded in CS781_Phase4_Six_Submissions_RunAll.ipynb.
Only the user's four Phase-1 checkpoints and official Phase-4 data are used.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
from collections import Counter, defaultdict
from contextlib import nullcontext
from datetime import timedelta
from pathlib import Path

import numpy as np
import torch
import torch.distributed as dist
from torch import nn
from torch.nn import functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.checkpoint import checkpoint

import cs781_phase4_utils as u

WORK = Path('/kaggle/working')
ROOT = WORK/'CS781_Capstone/phase4_cjpe'
ARTIFACT = WORK/'CS781_Phase4_All_Artifacts.zip'
u.ROOT = ROOT
u.ARTIFACT = ARTIFACT
SEED = 781404
REPO = 'Exploration-Lab/CS-781-Capstone'
SPLITS = {'train':'Train/train.json', 'dev':'Train/dev.json', 'test':'Test/test.json'}
SLOTS = {
    '01': {'base':'01','method':'full','epochs':9,'min_epochs':4,'lr_base':8e-6},
    '02': {'base':'03','method':'partial','epochs':10,'min_epochs':4,'lr_base':1.4e-5},
    '03': {'base':'04','method':'lorma','epochs':12,'min_epochs':5,'lr_base':5e-5},
    '04': {'base':'02','method':'adalora','epochs':12,'min_epochs':5,'lr_base':5e-5},
    '05': {'base':'03','method':'adapters','epochs':12,'min_epochs':5,'lr_base':5e-5},
    '06': {'base':'03','method':'distill','epochs':10,'min_epochs':4,'lr_base':1.2e-5},
}
FOLDS = 3
MAX_TOKENS = 640
MAX_CHUNKS = 8
OVERLAP = 64
ACCUM = 4
PATIENCE = 3
THRESHOLDS = tuple(round(x/100,2) for x in range(30,71,5))


def log(*parts):
    print(time.strftime('%Y-%m-%d %H:%M:%S'), *parts, flush=True)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed % (2**32-1))
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def stage_base(base_id):
    """Find a Phase-1 recovery by its contents, regardless of Kaggle's mount name."""
    dest = ROOT/'bases'/base_id
    needed = [f'model/{u.SOURCES[base_id]}','model/course_bpe.py',
              'checkpoints/selected_model.pt','tokenizer/tokenizer.json',
              'tokenizer/tokenizer_metadata.json']
    if all((dest/name).is_file() for name in needed): return dest
    expected = f'phase1submission{base_id}recovery'
    normalize = lambda value: ''.join(c for c in value.lower() if c.isalnum())
    candidates=[]
    for source in u.INPUT.rglob(u.SOURCES[base_id]):
        if source.parent.name!='model': continue
        base=source.parent.parent
        if all((base/name).is_file() for name in needed):
            priority=0 if expected in normalize(str(base)) else 2
            candidates.append((priority,'directory',base,''))
    archives=sorted(p for p in u.INPUT.rglob('*') if p.is_file() and p.suffix.lower()=='.zip')
    for archive in archives:
        try:
            with zipfile.ZipFile(archive) as bundle:
                names=set(bundle.namelist())
                suffix=f'model/{u.SOURCES[base_id]}'
                for member in names:
                    if not member.endswith(suffix):continue
                    prefix=member[:-len(suffix)]
                    if all(prefix+name in names for name in needed):
                        if not all(u.safe_name(prefix+name) for name in needed):continue
                        priority=1 if expected in normalize(str(archive)) else 3
                        candidates.append((priority,'archive',archive,prefix))
        except (OSError,zipfile.BadZipFile):continue
    if not candidates:
        mounts=sorted(p.name for p in u.INPUT.iterdir()) if u.INPUT.is_dir() else []
        files=sorted(str(p.relative_to(u.INPUT)) for p in u.INPUT.rglob(u.SOURCES[base_id]))
        raise RuntimeError(
            f'Phase-1 base {base_id} is unavailable in the attached Kaggle Inputs. '
            f'Expected a complete recovery containing {needed}. '
            f'Mounted inputs: {mounts}. Model-file matches: {files}. '
            f'ZIP files: {[str(p.relative_to(u.INPUT)) for p in archives]}. '
            f'Attach the Phase-1 submission {base_id} recovery dataset and Run All again.')
    candidates.sort(key=lambda row:(row[0],str(row[2]),row[3]))
    selected=[row for row in candidates if row[0]==candidates[0][0]]
    if len(selected)!=1:
        raise RuntimeError(f'Ambiguous Phase-1 base {base_id} recoveries: '
                           f'{[(kind,str(path),prefix) for _,kind,path,prefix in selected]}')
    _,kind,source,prefix=selected[0]
    if kind=='directory':
        wanted=set(needed)|{p.relative_to(source).as_posix() for p in (source/'tokenizer').rglob('*') if p.is_file()}
        def copy_files():
            for name in sorted(wanted):
                target=dest/name
                if target.is_file():continue
                target.parent.mkdir(parents=True,exist_ok=True)
                with tempfile.NamedTemporaryFile(dir=target.parent,delete=False) as handle:
                    temporary=Path(handle.name)
                    with (source/name).open('rb') as reader:shutil.copyfileobj(reader,handle)
                os.replace(temporary,target)
        copy_files()
    else:
        with zipfile.ZipFile(source) as bundle:
            wanted=set(needed)|{name[len(prefix):] for name in bundle.namelist()
                                if name.startswith(prefix+'tokenizer/') and not name.endswith('/')}
            for name in sorted(wanted):
                if not u.safe_name(name) or prefix+name not in bundle.namelist():
                    raise RuntimeError(f'Unsafe or missing Phase-1 member: {name}')
                target=dest/name
                if target.is_file():continue
                target.parent.mkdir(parents=True,exist_ok=True)
                with bundle.open(prefix+name) as reader,tempfile.NamedTemporaryFile(dir=target.parent,delete=False) as handle:
                    temporary=Path(handle.name);shutil.copyfileobj(reader,handle)
                os.replace(temporary,target)
    if not all((dest/name).is_file() for name in needed):
        raise RuntimeError(f'Phase-1 base {base_id} extraction was incomplete from {source}')
    u.write_json(dest/'provenance.json',{'mounted_source':str(source),'kind':kind,'member_prefix':prefix,
                 'model_sha256':u.sha(dest/needed[0]),
                 'checkpoint_sha256':u.sha(dest/'checkpoints/selected_model.pt')})
    log('STAGED PHASE-1 BASE',base_id,'from',source,'kind',kind)
    return dest


def stage_official():
    dest=ROOT/'official'
    if (dest/'provenance.json').is_file() and all((dest/f'{name}.json').is_file() for name in SPLITS):
        recorded=u.read_json(dest/'provenance.json')['sha256']
        if all(recorded.get(name)==u.sha(dest/f'{name}.json') for name in SPLITS):return
    try:
        from kaggle_secrets import UserSecretsClient
        token=UserSecretsClient().get_secret('HF_TOKEN')
        if not token: raise RuntimeError('HF_TOKEN is empty')
    except Exception as exc:
        raise RuntimeError('Add the enabled Kaggle Secret HF_TOKEN and turn Internet on') from exc
    try:
        from huggingface_hub import HfApi,hf_hub_download
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','huggingface_hub>=0.24,<2'])
        from huggingface_hub import HfApi,hf_hub_download
    api=HfApi()
    try:
        revision=api.repo_info(REPO,repo_type='dataset',token=token).sha
        paths=api.list_repo_files(REPO,repo_type='dataset',revision=revision,token=token)
    except Exception as exc:
        raise RuntimeError('Cannot access the official gated Phase-4 dataset. Check access, HF_TOKEN, Internet, and release date.') from exc
    chosen={}
    for name,suffix in SPLITS.items():
        hits=[path for path in paths if path.lower().endswith(('phase_4_cjpe/'+suffix).lower())]
        if len(hits)!=1:
            raise RuntimeError(f'Expected one official Phase-4 {suffix}; found {hits}. Dataset may not be released yet.')
        chosen[name]=hits[0]
    dest.mkdir(parents=True,exist_ok=True)
    for name,remote in chosen.items():
        target=dest/f'{name}.json'
        cached=Path(hf_hub_download(repo_id=REPO,repo_type='dataset',filename=remote,
                    revision=revision,token=token,cache_dir=str(ROOT/'hf_cache')))
        with tempfile.NamedTemporaryFile(dir=dest,delete=False) as handle:
            temporary=Path(handle.name)
            with cached.open('rb') as reader: shutil.copyfileobj(reader,handle)
        os.replace(temporary,target)
    u.write_json(dest/'provenance.json',{'repo':REPO,'revision':revision,'paths':chosen,
                'sha256':{name:u.sha(dest/f'{name}.json') for name in SPLITS}})
    del token


def parse_split(name):
    data = u.read_json(ROOT / 'official' / f'{name}.json')

    # ---------------------------------------------------------
    # Normalize supported top-level JSON layouts
    # ---------------------------------------------------------
    if isinstance(data, dict):
        for container_key in ('data', 'examples', 'cases'):
            if isinstance(data.get(container_key), list):
                data = data[container_key]
                break
        else:
            normalized = []

            for key, value in data.items():
                if isinstance(value, dict):
                    item = dict(value)
                    item.setdefault('id', str(key))
                else:
                    item = {
                        'id': str(key),
                        'text': value
                    }

                normalized.append(item)

            data = normalized

    if not isinstance(data, list):
        raise ValueError(
            f'{name}.json must contain a list of cases'
        )

    rows = []

    label_keys = (
        'label',
        'gold',
        'y',
        'outcome',
        'judgment',
        'judgement',
    )

    # ---------------------------------------------------------
    # Process every case
    # ---------------------------------------------------------
    for index, row in enumerate(data):

        if not isinstance(row, dict):
            raise ValueError(
                f'{name}: row {index} is not an object'
            )

        # -----------------------------------------------------
        # ID
        # -----------------------------------------------------
        case_id = row.get(
            'id',
            row.get('case_id')
        )

        if case_id is None:
            raise ValueError(
                f'{name}: row {index} has no id/case_id'
            )

        case_id = str(case_id).strip()

        if not case_id:
            raise ValueError(
                f'{name}: row {index} has an empty id'
            )

        # -----------------------------------------------------
        # Text
        # -----------------------------------------------------
        content = row.get(
            'text',
            row.get('case_text')
        )

        if isinstance(content, list):
            content = '\n'.join(
                str(part)
                for part in content
            )
        elif content is not None and not isinstance(content, str):
            content = str(content)

        if not isinstance(content, str) or not content.strip():
            raise ValueError(
                f'{name}: case {case_id!r} has missing/empty text'
            )

        # =====================================================
        # TEST SPLIT
        # =====================================================
        if name == 'test':
            # The official test file may contain a label field.
            # Ignore it completely so test labels cannot affect
            # training, validation, thresholding, or model choice.
            label = None

        # =====================================================
        # TRAIN / DEV SPLITS
        # =====================================================
        else:
            binary_candidates = []

            for key in label_keys:
                if key not in row:
                    continue

                value = row[key]

                if isinstance(value, bool):
                    binary_candidates.append((key, int(value)))

                elif isinstance(value, int) and value in (0, 1):
                    binary_candidates.append((key, int(value)))

                elif isinstance(value, str) and value.strip() in ('0', '1'):
                    binary_candidates.append((key, int(value.strip())))

            if not binary_candidates:
                raise ValueError(
                    f'{name}: no binary label found for case {case_id!r}; '
                    f'available keys={sorted(row.keys())}'
                )

            values = {
                value
                for _, value in binary_candidates
            }

            if len(values) != 1:
                raise ValueError(
                    f'{name}: conflicting binary labels for case '
                    f'{case_id!r}: {binary_candidates}'
                )

            label = values.pop()

        # -----------------------------------------------------
        # Internal IDs
        # -----------------------------------------------------
        if name == 'test':
            internal_id = case_id
        else:
            internal_id = f'{name}::row::{index:08d}'

        rows.append(
            {
                'id': internal_id,
                'source_id': case_id,
                'text': content,
                'label': label,
            }
        )

    # ---------------------------------------------------------
    # Final sanity checks
    # ---------------------------------------------------------
    if not rows:
        raise ValueError(
            f'Empty {name}.json'
        )

    ids = [
        row['id']
        for row in rows
    ]

    if len(ids) != len(set(ids)):
        raise ValueError(
            f'Duplicate internal IDs in {name}.json'
        )

    return rows

def data():
    train,dev,test=(parse_split(name) for name in ('train','dev','test'))
    groups=[set(row['id'] for row in split) for split in (train,dev,test)]
    if any(groups[i]&groups[j] for i in range(3) for j in range(i+1,3)):
        raise ValueError('Official Phase-4 case IDs overlap across splits')
    if set(row['label'] for row in train)!={0,1} or set(row['label'] for row in dev)!={0,1}:
        raise ValueError('Both labels must occur in the train and dev splits for macro F1')
    return train,dev,test


def text_hash(text):
    return hashlib.sha256(re.sub(r'\s+',' ',text.lower()).strip().encode('utf8')).hexdigest()


def case_groups(rows,tie_source=True):
    """Keep repeated case IDs and exact duplicate texts in one validation group."""
    parent=list(range(len(rows)))
    def root(i):
        while parent[i]!=i:
            parent[i]=parent[parent[i]]
            i=parent[i]
        return i
    def merge(i,j):parent[root(i)]=root(j)
    seen={}
    for i,row in enumerate(rows):
        keys=[('text',text_hash(row['text']))]
        if tie_source:keys.append(('case',row['source_id']))
        for key in keys:
            if key in seen:merge(i,seen[key])
            else:seen[key]=i
    grouped=defaultdict(list)
    for i,row in enumerate(rows):grouped[root(i)].append(row)
    return list(grouped.values())


def validation_groups(rows,force_mode=None):
    if force_mode is not None:
        if force_mode not in ('case_id_and_exact_text','exact_text_only'):
            raise ValueError(f'Unknown validation grouping mode: {force_mode}')
        return case_groups(rows,tie_source=force_mode=='case_id_and_exact_text'),force_mode
    grouped=case_groups(rows,tie_source=True)
    # A generic reused ID can join a large fraction of unrelated cases.
    # In that situation, exact-text grouping is the defensible split unit.
    if len(grouped)<FOLDS*2 or max(map(len,grouped))>max(12,len(rows)//4):
        log('WARNING: source case IDs are too coarse for grouped folds; grouping by exact text only',
            'rows',len(rows),'source groups',len(grouped),'largest',max(map(len,grouped)))
        return case_groups(rows,tie_source=False),'exact_text_only'
    return grouped,'case_id_and_exact_text'


def folds_for(train,return_mode=False):
    groups,first_mode=validation_groups(train)
    modes=[first_mode]+(['exact_text_only'] if first_mode!='exact_text_only' else [])
    for mode in modes:
        groups,_=validation_groups(train,mode)
        buckets=[[],[]]
        for group in groups:
            label=int(sum(row['label'] for row in group)*2>=len(group))
            buckets[label].append(group)
        mapping={};counts=[[0,0] for _ in range(FOLDS)]
        for label,items in enumerate(buckets):
            random.Random(SEED+label).shuffle(items)
            items.sort(key=lambda group:-len(group))
            for group in items:
                added=[sum(row['label']==class_id for row in group) for class_id in (0,1)]
                fold=min(range(FOLDS),key=lambda i:(counts[i][label],sum(counts[i]),i))
                for row in group:mapping[row['id']]=fold
                for class_id in (0,1):counts[fold][class_id]+=added[class_id]
        if all(all(counts[i]) for i in range(FOLDS)):
            if mode!=first_mode:log('WARNING: source IDs prevent class-balanced folds; used exact-text grouping')
            return (mapping,mode) if return_mode else mapping
    raise ValueError('Too few independent cases of one class for three grouped outer folds')


def inner_split(rows,seed,group_mode=None):
    groups,_=validation_groups(rows,group_mode)
    by_label=[[],[]]
    for items in groups:
        label=int(sum(row['label'] for row in items)*2>=len(items))
        by_label[label].append(items)
    inner=[]
    for label,items in enumerate(by_label):
        random.Random(seed+label).shuffle(items)
        n=max(1,round(.15*sum(len(group) for group in items)))
        count=0
        for group in items:
            if count>=n:break
            inner.extend(group);count+=len(group)
    selected={row['id'] for row in inner}
    fit=[row for row in rows if row['id'] not in selected]
    if not fit or {r['label'] for r in fit}!={0,1} or {r['label'] for r in inner}!={0,1}:
        raise ValueError('Cannot form a stratified inner validation split')
    return fit,inner


def prepare():
    if torch.cuda.device_count()!=2:raise RuntimeError('Enable the Kaggle accelerator with exactly two GPUs')
    ROOT.mkdir(parents=True,exist_ok=True)
    u.restore_artifact()
    for base in ('01','02','03','04'):stage_base(base)
    stage_official()
    train,dev,test=data()
    for split_name,rows in (('train',train),('dev',dev),('test',test)):
        id_counts=Counter(row['source_id'] for row in rows)
        log('CASE-ID AUDIT',split_name,'rows',len(rows),'unique original IDs',len(id_counts),
            'repeated rows',len(rows)-len(id_counts),'largest ID group',max(id_counts.values()))
    foldmap,group_mode=folds_for(train,return_mode=True)
    hashes={name:u.sha(ROOT/'official'/f'{name}.json') for name in SPLITS}
    config={'version':2,'source_sha256':u.sha(__file__),'seed':SEED,'folds':foldmap,
            'validation_group_mode':group_mode,
            'official_sha256':hashes,'max_tokens':MAX_TOKENS,'max_chunks':MAX_CHUNKS,
            'slots':SLOTS}
    target=ROOT/'run_config.json'
    if target.is_file() and u.read_json(target)!=config:
        raise RuntimeError('Recovery archive belongs to different code, data, or configuration')
    u.write_json(target,config)
    shutil.copy2(__file__,ROOT/'cjpe_runner.py')
    shutil.copy2(Path(u.__file__),ROOT/'phase1_utils.py')
    counts={name:dict(Counter(row['label'] for row in rows)) for name,rows in (('train',train),('dev',dev))}
    duplicates={name:len(rows)-len({text_hash(r['text']) for r in rows}) for name,rows in (('train',train),('dev',dev),('test',test))}
    repeated_ids={name:len(rows)-len({r['source_id'] for r in rows})
                  for name,rows in (('train',train),('dev',dev),('test',test))}
    lengths={name:{'count':len(rows),'characters_median':int(np.median([len(r['text']) for r in rows])),
                   'characters_p95':int(np.percentile([len(r['text']) for r in rows],95))}
             for name,rows in (('train',train),('dev',dev),('test',test))}
    baseline=max(counts['dev'].values())/(sum(counts['dev'].values()))
    validation_group_list,validation_group_mode=validation_groups(train,group_mode)
    u.write_json(ROOT/'data_audit.json',{'label_counts':counts,'lengths':lengths,
                 'exact_duplicate_counts':duplicates,'repeated_source_id_rows':repeated_ids,
                 'case_groups_train':len(validation_group_list),'validation_group_mode':validation_group_mode,
                 'majority_accuracy_dev':baseline,
                 'train_case_count':len(train),'dev_case_count':len(dev),'test_case_count':len(test),
                 'test_id_sha256':hashlib.sha256('\n'.join(sorted(r['id'] for r in test)).encode()).hexdigest(),
                 'private_score':None})
    log('PREPARED',len(train),'train',len(dev),'dev',len(test),'test','label counts',counts)


def tokenize_base(base_id):
    target=ROOT/'hf_cache'/f'tokens_{base_id}.pt'
    if target.is_file():return
    _,tokenizer=u.load_base(base_id)
    train,dev,test=data()
    rows=train+dev+test
    encoded={row['id']:tokenizer.backend.encode(row['text'],add_special_tokens=False).ids for row in rows}
    marker=tokenizer.backend.encode(' [JUDGMENT OUTCOME]',add_special_tokens=False).ids
    if not marker:raise RuntimeError('Outcome marker is empty')
    target.parent.mkdir(parents=True,exist_ok=True)
    u.save_torch(target,{'tokens':encoded,'marker':marker,'pad':int(tokenizer.pad_token_id)})
    lengths={name:[len(encoded[row['id']]) for row in split] for name,split in
             zip(('train','dev','test'),(train,dev,test))}
    u.write_json(ROOT/f'token_audit_{base_id}.json',{
        name:{'median':int(np.median(values)),'p95':int(np.percentile(values,95)),
              'max':max(values),'over_8_chunks':sum(v>MAX_CHUNKS*(MAX_TOKENS-OVERLAP) for v in values)}
        for name,values in lengths.items()})
    log('TOKENIZED',base_id,'cases',len(rows))


def chunk_starts(size,width,training,epoch,case_id):
    if size<=width:return [0]
    last=size-width
    count=min(MAX_CHUNKS,math.ceil(size/max(1,width-OVERLAP)))
    if count==1:return [0]
    values=[round(last*i/(count-1)) for i in range(count)]
    if training and count==MAX_CHUNKS and last>width:
        rng=random.Random(SEED+epoch*104729+int(hashlib.sha256(case_id.encode()).hexdigest()[:8],16))
        stride=last/(count-1)
        for i in range(1,count-1):values[i]=min(last,max(0,round(values[i]+rng.uniform(-.25,.25)*stride)))
    return sorted(set([0,last]+values))


def document_batch(row,cache,device,training=False,epoch=0):
    tokens=cache['tokens'][row['id']]
    marker=cache['marker']
    width=MAX_TOKENS-len(marker)
    if width<128:raise ValueError('Outcome marker unexpectedly long')
    starts=chunk_starts(len(tokens),width,training,epoch,row['id'])
    chunks=[(tokens[start:start+width] or [cache['pad']])+marker for start in starts]
    length=max(map(len,chunks))
    ids=torch.full((len(chunks),length),cache['pad'],dtype=torch.long)
    mask=torch.zeros_like(ids,dtype=torch.bool)
    for i,chunk in enumerate(chunks):
        ids[i,:len(chunk)]=torch.tensor(chunk,dtype=torch.long)
        mask[i,:len(chunk)]=True
    position=torch.tensor([min(31,round(31*start/max(1,len(tokens)-width))) for start in starts],dtype=torch.long)
    if device.type=='cuda':
        ids=ids.pin_memory().to(device,non_blocking=True)
        mask=mask.pin_memory().to(device,non_blocking=True)
        position=position.pin_memory().to(device,non_blocking=True)
    return ids,mask,position


def macro_f1(probs,labels,threshold=.5):
    if len(probs)!=len(labels) or not probs:raise ValueError('Empty or mismatched score set')
    pred=[int(value>=threshold) for value in probs]
    scores=[];details={}
    for label in (0,1):
        tp=sum(p==label and y==label for p,y in zip(pred,labels))
        fp=sum(p==label and y!=label for p,y in zip(pred,labels))
        fn=sum(p!=label and y==label for p,y in zip(pred,labels))
        precision=tp/(tp+fp) if tp+fp else 0.
        recall=tp/(tp+fn) if tp+fn else 0.
        f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.
        details[str(label)]={'precision':precision,'recall':recall,'f1':f1,'tp':tp,'fp':fp,'fn':fn,
                             'gold':sum(y==label for y in labels),'predicted':sum(p==label for p in pred)}
        scores.append(f1)
    return {'macro_f1':sum(scores)/2,'per_class':details,'threshold':threshold,
            'accuracy':sum(p==y for p,y in zip(pred,labels))/len(labels)}


def best_threshold(probs,labels):
    values=[macro_f1(probs,labels,t) for t in THRESHOLDS]
    return max(values,key=lambda result:(result['macro_f1'],-abs(result['threshold']-.5)))


class BottleneckAdapter(nn.Module):
    def __init__(self,block,width,rank=96):
        super().__init__()
        self.block=block
        self.down=nn.Linear(width,rank)
        self.up=nn.Linear(rank,width)
        nn.init.zeros_(self.up.weight);nn.init.zeros_(self.up.bias)
        self.scale=nn.Parameter(torch.tensor(1.))

    def forward(self,*args):
        result=self.block(*args)
        return result+self.scale*self.up(F.gelu(self.down(result)))


class AdaLoRALinear(nn.Module):
    """Low-rank SVD-like update with gradient-importance rank allocation."""
    def __init__(self,base,rank=16,alpha=16):
        super().__init__()
        self.base=base
        self.a=nn.Linear(base.in_features,rank,bias=False)
        self.b=nn.Linear(rank,base.out_features,bias=False)
        self.s=nn.Parameter(torch.full((rank,),.01))
        self.register_buffer('mask',torch.ones(rank))
        self.register_buffer('importance',torch.zeros(rank))
        nn.init.kaiming_uniform_(self.a.weight,a=math.sqrt(5))
        nn.init.normal_(self.b.weight,std=.01)
        self.scale=alpha/rank

    def forward(self,x):
        return self.base(x)+self.b(self.a(x)*(self.s*self.mask))*self.scale

    @torch.no_grad()
    def record_importance(self):
        if self.s.grad is not None:
            self.importance.mul_(.9).add_((self.s*self.s.grad).abs(),alpha=.1)


def allocate_ada_ranks(model,epoch,total_epochs):
    modules=[module for module in model.modules() if isinstance(module,AdaLoRALinear)]
    if not modules:return
    initial=sum(module.mask.numel() for module in modules)
    target=max(4*len(modules),round(initial*(1.-.5*min(1.,epoch/max(1,total_epochs)))))
    candidates=[]
    for i,module in enumerate(modules):
        ranking=torch.argsort(module.importance,descending=True).tolist()
        for j in ranking[:4]:candidates.append((float('inf'),i,j))
        for j in ranking[4:]:candidates.append((float(module.importance[j]),i,j))
    candidates.sort(key=lambda row:row[0],reverse=True)
    selected={(i,j) for _,i,j in candidates[:target]}
    for i,module in enumerate(modules):
        module.mask.copy_(torch.tensor([float((i,j) in selected) for j in range(module.mask.numel())],
                                       device=module.mask.device,dtype=module.mask.dtype))


class CaseClassifier(nn.Module):
    def __init__(self,base,base_id,method):
        super().__init__()
        self.base=base
        self.base_id=base_id
        self.method=method
        for p in base.parameters():p.requires_grad_(False)
        self.first_trainable=0
        if base_id=='01':width=int(base.config.hidden_size)
        elif base_id=='04':width=int(base.config['hidden_size'])
        else:width=int(base.hidden_size)
        if method=='full':
            for p in base.parameters():p.requires_grad_(True)
        elif method in ('partial','distill'):
            self.first_trainable=len(base.blocks)-(12 if method=='partial' else 8)
            for block in base.blocks[self.first_trainable:]:
                for p in block.parameters():p.requires_grad_(True)
            for p in base.final_norm.parameters():p.requires_grad_(True)
        elif method=='lorma':
            count=0
            for name,module in list(base.named_modules()):
                if (isinstance(module,nn.Linear) and
                    name.split('.')[-1] in ('q_proj','v_proj','o_proj','gate_proj','up_proj') and
                    name.startswith('blocks.')):
                    parent=base.get_submodule(name.rsplit('.',1)[0])
                    setattr(parent,name.rsplit('.',1)[-1],u.LoRMA(module,16,16))
                    count+=1
            if count<14:raise RuntimeError('LORMA did not adapt base 04 blocks')
        elif method=='adalora':
            count=0
            for name,module in list(base.named_modules()):
                if (name.startswith('encoder_blocks.') and isinstance(module,nn.Linear) and
                    name.split('.')[-1] in ('query','value','input','output')):
                    parent=base.get_submodule(name.rsplit('.',1)[0])
                    setattr(parent,name.rsplit('.',1)[-1],AdaLoRALinear(module))
                    count+=1
            if count<12:raise RuntimeError('AdaLoRA did not adapt base 02 encoder')
            base.set_gradient_checkpointing(True)
        elif method=='adapters':
            self.first_trainable=len(base.blocks)-12
            for i in range(self.first_trainable,len(base.blocks)):
                base.blocks[i]=BottleneckAdapter(base.blocks[i],width)
        else:raise ValueError(method)
        if base_id=='04':base.set_gradient_checkpointing(True)
        self.project=nn.Sequential(nn.Linear(width,256),nn.LayerNorm(256),nn.GELU(),nn.Dropout(.15))
        self.position=nn.Embedding(32,256)
        self.context=nn.TransformerEncoder(
            nn.TransformerEncoderLayer(256,4,512,dropout=.15,batch_first=True,norm_first=True),
            num_layers=1,enable_nested_tensor=False)
        self.gate=nn.Linear(256,1)
        self.classifier=nn.Sequential(nn.LayerNorm(768),nn.Dropout(.25),nn.Linear(768,256),
                                      nn.GELU(),nn.Dropout(.2),nn.Linear(256,1))

    def chunk_features(self,ids,mask):
        base=self.base
        if self.base_id=='01':
            positions=torch.arange(ids.size(1),device=ids.device)
            states=base.embedding_dropout(base.token_embedding(ids)+base.position_embedding(positions)[None])
            for block in base.blocks:
                states=checkpoint(block,states,use_reentrant=False) if self.training else block(states)
            states=base.final_norm(states)
        elif self.base_id=='02':
            states,_=base.encode(ids,mask)
        elif self.base_id=='03':
            states=base.embedding_dropout(base.token_embedding(ids)*math.sqrt(base.hidden_size))
            visibility=base._prefix_hybrid_visibility(mask,mask.sum(1))
            cosine,sine=base.rotary(ids.size(1),states.dtype)
            for i,block in enumerate(base.blocks):
                if i<self.first_trainable:
                    with torch.no_grad():states=block(states,visibility,cosine,sine)
                else:
                    states=checkpoint(block,states,visibility,cosine,sine,use_reentrant=False) if self.training else block(states,visibility,cosine,sine)
            states=base.final_norm(states)
        else:
            states=base.embedding(ids)
            for block in base.blocks:
                states=checkpoint(block,states,mask,use_reentrant=False) if self.training else block(states,mask)
            states=base.final_norm(states)
        if self.base_id in ('01','04'):
            end=mask.sum(1).long()-1
            return states[torch.arange(ids.size(0),device=ids.device),end]
        return (states*mask.unsqueeze(-1)).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)

    def forward(self,ids,mask,position):
        chunks=self.project(self.chunk_features(ids,mask))+self.position(position)
        chunks=self.context(chunks[None])[0]
        weight=self.gate(chunks).squeeze(-1).softmax(0)
        pooled=(chunks*weight[:,None]).sum(0)
        combined=torch.cat((pooled,chunks.mean(0),chunks.max(0).values),-1)
        return self.classifier(combined).squeeze(-1)


def model_for(slot,device):
    spec=SLOTS[slot]
    base,tokenizer=u.load_base(spec['base'])
    model=CaseClassifier(base,spec['base'],spec['method']).to(device)
    return model,tokenizer


def trainable_state(model):
    keys={name for name,p in model.named_parameters() if p.requires_grad}
    buffers={name for name,_ in model.named_buffers() if name.endswith(('.mask','.importance'))}
    return {name:value.detach().cpu().clone() for name,value in model.state_dict().items()
            if name in keys or name in buffers}


def load_state(model,state):
    keys={name for name,p in model.named_parameters() if p.requires_grad}
    buffers={name for name,_ in model.named_buffers() if name.endswith(('.mask','.importance'))}
    expected=keys|buffers
    if set(state)!=expected:
        raise RuntimeError(f'Trainable-state mismatch: missing={expected-set(state)} extra={set(state)-expected}')
    model.load_state_dict(state,strict=False)


def optimizer_for(model,spec,updates):
    groups={}
    for name,p in model.named_parameters():
        if not p.requires_grad:continue
        if name.startswith('base.'):
            lr=spec['lr_base']
            if '.a.' in name or '.b.' in name or '.down.' in name or '.up.' in name or '.s' in name:
                lr=max(lr,5e-5)
        else:lr=1.2e-4
        decay=0. if name.endswith('bias') or 'norm' in name.lower() or name.endswith('.s') else .01
        groups.setdefault((lr,decay),[]).append(p)
    parameters=[{'params':values,'lr':lr,'weight_decay':decay} for (lr,decay),values in groups.items()]
    try:optimizer=torch.optim.AdamW(parameters,fused=True)
    except (TypeError,RuntimeError):optimizer=torch.optim.AdamW(parameters)
    warmup=max(4,round(.06*updates))
    def schedule(step):
        if step<warmup:return max(.1,(step+1)/warmup)
        portion=min(1.,(step-warmup)/max(1,updates-warmup))
        return .12+.88*.5*(1+math.cos(math.pi*portion))
    return optimizer,torch.optim.lr_scheduler.LambdaLR(optimizer,schedule)


def init_dist():
    if int(os.environ.get('WORLD_SIZE','1'))!=2 or torch.cuda.device_count()!=2:
        raise RuntimeError('Use torchrun --standalone --nproc_per_node=2 with two visible GPUs')
    local=int(os.environ['LOCAL_RANK'])
    torch.cuda.set_device(local)
    torch.set_num_threads(2)
    torch.backends.cuda.matmul.allow_tf32=True
    device=torch.device('cuda',local)
    try:dist.init_process_group('nccl',timeout=timedelta(minutes=45),device_id=device)
    except TypeError:dist.init_process_group('nccl',timeout=timedelta(minutes=45))
    return int(os.environ['RANK']),device


def stage_rows(stage):
    train,dev,test=data()
    if stage=='final':return train,dev,test
    if not stage.startswith('fold_'):raise ValueError(stage)
    fold=int(stage.split('_')[1])
    if fold not in range(FOLDS):raise ValueError(stage)
    config=u.read_json(ROOT/'run_config.json')
    mapping=config['folds']
    outer=[row for row in train if mapping[row['id']]==fold]
    available=[row for row in train if mapping[row['id']]!=fold]
    fit,inner=inner_split(available,SEED+fold*103,config.get('validation_group_mode'))
    if not outer or not fit or not inner:raise RuntimeError('Empty fold partition')
    return fit,inner,outer


def epoch_order(rows,slot,stage,epoch):
    ids=[row['id'] for row in rows]
    seed=SEED+sum(map(ord,slot+stage))*997+epoch*17
    rng=random.Random(seed)
    rng.shuffle(ids)
    original=list(ids)
    while len(ids)%(2*ACCUM):ids.append(original[(len(ids)-len(original))%len(original)])
    return ids


@torch.no_grad()
def predict_rows(model,rows,cache,rank,device):
    model.eval()
    local={}
    for row in rows[rank::2]:
        ids,mask,pos=document_batch(row,cache,device,False)
        with torch.autocast('cuda',dtype=torch.float16):
            value=model(ids,mask,pos)
        local[row['id']]=float(value.float().cpu())
    gathered=[None,None]
    dist.all_gather_object(gathered,local)
    result={key:value for piece in gathered for key,value in piece.items()}
    if set(result)!={row['id'] for row in rows}:
        raise RuntimeError('Distributed inference did not cover each case exactly once')
    return result


def teacher_cache(stage,fit,rank,device):
    destination=ROOT/'runs'/'06'/stage/'teacher_logits.pt'
    if destination.is_file():
        values=torch.load(destination,map_location='cpu',weights_only=False)
        if set(values)!={row['id'] for row in fit}:raise RuntimeError('Teacher cache ID mismatch')
        return values
    teacher_path=ROOT/'runs'/'01'/stage/'best.pt'
    if not teacher_path.is_file():raise RuntimeError('Train the matching base-01 teacher stage before distillation')
    teacher,tokenizer=model_for('01',device)
    load_state(teacher,torch.load(teacher_path,map_location='cpu',weights_only=False)['model'])
    token_path=ROOT/'hf_cache/tokens_01.pt'
    if not token_path.is_file():raise FileNotFoundError(token_path)
    cache=torch.load(token_path,map_location='cpu',weights_only=False)
    result=predict_rows(teacher,fit,cache,rank,device)
    if rank==0:
        destination.parent.mkdir(parents=True,exist_ok=True)
        u.save_torch(destination,result)
        log('CACHED FOLD-SAFE TEACHER',stage,len(result))
    dist.barrier()
    del teacher,tokenizer,cache
    torch.cuda.empty_cache()
    return result


def rng_state():
    return {'python':random.getstate(),'numpy':np.random.get_state(),
            'torch':torch.get_rng_state(),'cuda':torch.cuda.get_rng_state()}


def restore_rng(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'])
    torch.cuda.set_rng_state(state['cuda'])


def gpu_sample(slot,stage,epoch):
    try:
        process=subprocess.run(['nvidia-smi','--query-gpu=index,utilization.gpu,memory.used,memory.total',
              '--format=csv,noheader,nounits'],capture_output=True,text=True,check=True,timeout=10)
        rows=[line.strip() for line in process.stdout.splitlines()]
    except Exception as exc:rows=[repr(exc)]
    with (ROOT/'gpu_samples.jsonl').open('a',encoding='utf8') as handle:
        handle.write(json.dumps({'slot':slot,'stage':stage,'epoch':epoch,'gpus':rows,'time':time.time()})+'\n')


def train_stage(slot,stage,chunk_epochs):
    rank,device=init_dist()
    try:
        spec=SLOTS[slot]
        directory=ROOT/'runs'/slot/stage
        directory.mkdir(parents=True,exist_ok=True)
        status=directory/'status.json'
        if status.is_file() and u.read_json(status).get('complete'):
            if rank==0:log('REUSE',slot,stage)
            dist.barrier();return
        seed_all(SEED+int(slot)*1000+sum(map(ord,stage))+rank)
        fit,inner,outer=stage_rows(stage)
        cache_path=ROOT/'hf_cache'/f"tokens_{spec['base']}.pt"
        if not cache_path.is_file():raise FileNotFoundError(cache_path)
        cache=torch.load(cache_path,map_location='cpu',weights_only=False)
        by_id={row['id']:row for row in fit}
        teacher=teacher_cache(stage,fit,rank,device) if slot=='06' else None
        model,tokenizer=model_for(slot,device)
        steps_per_epoch=len(epoch_order(fit,slot,stage,1))//(2*ACCUM)
        optimizer,scheduler=optimizer_for(model,spec,max(1,steps_per_epoch*spec['epochs']))
        scaler=torch.amp.GradScaler('cuda')
        latest=directory/'latest.pt'
        history=u.read_json(directory/'history.json') if (directory/'history.json').is_file() else []
        start=1;best=-1.;best_epoch=0;best_threshold_value=.5
        state=None
        if latest.is_file():
            state=torch.load(latest,map_location='cpu',weights_only=False)
            load_state(model,state['model'])
            optimizer.load_state_dict(state['optimizer'])
            scheduler.load_state_dict(state['scheduler'])
            scaler.load_state_dict(state['scaler'])
            start=state['epoch']+1;best=state['best'];best_epoch=state['best_epoch']
            best_threshold_value=state['best_threshold']
            restore_rng(state['rng'][rank])
        ddp=DDP(model,device_ids=[device.index],broadcast_buffers=False,
                find_unused_parameters=False,gradient_as_bucket_view=True)
        count=Counter(row['label'] for row in fit)
        pos_weight=torch.tensor(min(2.,max(.5,count[0]/max(1,count[1]))),device=device)
        limit=min(spec['epochs'],start+chunk_epochs-1)
        finished=start>spec['epochs']
        for epoch in range(start,limit+1):
            epoch_started=time.monotonic()
            torch.cuda.reset_peak_memory_stats(device)
            ddp.train();optimizer.zero_grad(set_to_none=True)
            order=epoch_order(fit,slot,stage,epoch)[rank::2]
            loss_sum=0.;step_count=0;processed_tokens=0
            for j,case_id in enumerate(order):
                row=by_id[case_id]
                ids,mask,pos=document_batch(row,cache,device,True,epoch)
                processed_tokens+=int(mask.sum())
                sync=(j+1)%ACCUM==0
                with (nullcontext() if sync else ddp.no_sync()):
                    with torch.autocast('cuda',dtype=torch.float16):
                        value=ddp(ids,mask,pos)
                        target=torch.tensor(float(row['label']),device=device)
                        hard=F.binary_cross_entropy_with_logits(value.float(),target,pos_weight=pos_weight)
                        if teacher is None:loss=hard/ACCUM
                        else:
                            temperature=2.
                            soft=torch.sigmoid(torch.tensor(teacher[case_id]/temperature,device=device))
                            distill=F.binary_cross_entropy_with_logits(value.float()/temperature,soft)*temperature**2
                            loss=(.75*hard+.25*distill)/ACCUM
                    if not bool(torch.isfinite(loss)):raise FloatingPointError('Nonfinite CJPE loss')
                    scaler.scale(loss).backward()
                loss_sum+=float(loss.detach())*ACCUM
                if sync:
                    scaler.unscale_(optimizer)
                    for module in model.modules():
                        if isinstance(module,AdaLoRALinear):module.record_importance()
                    nn.utils.clip_grad_norm_(model.parameters(),1.)
                    scaler.step(optimizer);scaler.update();scheduler.step()
                    optimizer.zero_grad(set_to_none=True);step_count+=1
            if spec['method']=='adalora':allocate_ada_ranks(model,epoch,spec['epochs'])
            dist.barrier()
            validation=predict_rows(model,inner,cache,rank,device)
            probabilities=[float(torch.sigmoid(torch.tensor(validation[row['id']]))) for row in inner]
            observed=best_threshold(probabilities,[row['label'] for row in inner])
            if rank==0:
                score=observed['macro_f1']
                if score>best+1e-6:
                    best=score;best_epoch=epoch;best_threshold_value=observed['threshold']
                    u.save_torch(directory/'best.pt',{'model':trainable_state(model),'epoch':epoch,
                                   'inner_macro_f1':best,'threshold':best_threshold_value,
                                   'slot':slot,'stage':stage})
                history.append({'epoch':epoch,'train_loss_rank0':loss_sum/max(1,len(order)),
                                'inner':observed,'best_epoch':best_epoch,
                                'optimizer_steps_rank0':step_count,
                                'epoch_seconds':round(time.monotonic()-epoch_started,2),
                                'rank0_tokens_per_second':round(processed_tokens/max(1,time.monotonic()-epoch_started),1),
                                'rank0_peak_gpu_gib':round(torch.cuda.max_memory_allocated(device)/2**30,2)})
                u.write_json(directory/'history.json',history)
                log('TRAIN',slot,stage,'epoch',epoch,'loss',round(history[-1]['train_loss_rank0'],4),
                    'inner macro F1',round(score,4),'best',round(best,4),
                    'rank0 tokens/s',history[-1]['rank0_tokens_per_second'],
                    'peak GiB',history[-1]['rank0_peak_gpu_gib'])
                gpu_sample(slot,stage,epoch)
                finished=(epoch>=spec['min_epochs'] and epoch-best_epoch>=PATIENCE) or epoch>=spec['epochs']
                u.write_json(status,{'complete':False,'epoch':epoch,'best_epoch':best_epoch,
                                     'best_inner_macro_f1':best})
            flags=torch.tensor([int(finished)],device=device)
            dist.broadcast(flags,0);finished=bool(flags.item())
            states=[None,None]
            dist.all_gather_object(states,rng_state())
            if rank==0:
                u.save_torch(latest,{'model':trainable_state(model),'optimizer':optimizer.state_dict(),
                             'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),
                             'epoch':epoch,'best':best,'best_epoch':best_epoch,
                             'best_threshold':best_threshold_value,'rng':states})
            dist.barrier()
            if finished:break
        if finished:
            selected=torch.load(directory/'best.pt',map_location='cpu',weights_only=False)
            load_state(model,selected['model'])
            dev_predictions=predict_rows(model,inner,cache,rank,device) if stage=='final' else None
            predictions=predict_rows(model,outer,cache,rank,device)
            if rank==0:
                if dev_predictions is not None:u.save_torch(directory/'dev_logits.pt',dev_predictions)
                u.save_torch(directory/('test_logits.pt' if stage=='final' else 'outer_logits.pt'),predictions)
                u.write_json(status,{'complete':True,'epoch':history[-1]['epoch'],
                    'best_epoch':best_epoch,'inner_macro_f1':best,'threshold':best_threshold_value,
                    'predicted_cases':len(predictions)})
                latest.unlink(missing_ok=True)
                log('COMPLETE',slot,stage,'cases',len(predictions),'best epoch',best_epoch)
            dist.barrier()
    finally:
        dist.destroy_process_group()


def complete(slot,stage):
    target=ROOT/'runs'/slot/stage/'status.json'
    return target.is_file() and bool(u.read_json(target).get('complete'))


def summarize_slot(slot):
    if not complete(slot,'final'):
        raise RuntimeError(f'Slot {slot} final model is incomplete; resume training before packaging')
    train,dev,test=data()
    mapping=u.read_json(ROOT/'run_config.json')['folds']
    outer_logits={};thresholds=[];outer_predictions={};fold_metrics={}
    for fold in range(FOLDS):
        if not complete(slot,f'fold_{fold}'):continue
        path=ROOT/'runs'/slot/f'fold_{fold}'
        values=torch.load(path/'outer_logits.pt',map_location='cpu',weights_only=False)
        expected={row['id'] for row in train if mapping[row['id']]==fold}
        if set(values)!=expected or set(values)&set(outer_logits):
            raise RuntimeError(f'Slot {slot} fold {fold}: outer IDs wrong or repeated')
        threshold=float(u.read_json(path/'status.json')['threshold'])
        thresholds.append(threshold)
        outer_logits.update(values)
        outer_predictions.update({key:int(torch.sigmoid(torch.tensor(value)).item()>=threshold)
                                  for key,value in values.items()})
        heldout=[row for row in train if mapping[row['id']]==fold]
        fold_metrics[str(fold)]=macro_f1([outer_predictions[row['id']] for row in heldout],
                                          [row['label'] for row in heldout],.5)
    validated_rows=[row for row in train if row['id'] in outer_logits]
    outer=macro_f1([outer_predictions[row['id']] for row in validated_rows],
                   [row['label'] for row in validated_rows],.5) if validated_rows else None
    complete_outer=len(thresholds)==FOLDS
    if complete_outer and set(outer_logits)!={row['id'] for row in train}:
        raise RuntimeError(f'Slot {slot}: complete folds have incomplete OOF coverage')
    final_path=ROOT/'runs'/slot/'final'
    dev_logits=torch.load(final_path/'dev_logits.pt',map_location='cpu',weights_only=False)
    test_logits=torch.load(final_path/'test_logits.pt',map_location='cpu',weights_only=False)
    if set(dev_logits)!={row['id'] for row in dev} or set(test_logits)!={row['id'] for row in test}:
        raise RuntimeError(f'Slot {slot}: final dev/test IDs do not match official data')
    final_threshold=float(np.median(thresholds)) if len(thresholds)>=2 else float(u.read_json(final_path/'status.json')['threshold'])
    dev_probs=[float(torch.sigmoid(torch.tensor(dev_logits[row['id']]))) for row in dev]
    development=macro_f1(dev_probs,[row['label'] for row in dev],final_threshold)
    labels=[row['label'] for row in train]
    majority=int(Counter(labels)[1]>Counter(labels)[0])
    majority_macro=macro_f1([float(majority)]*len(labels),labels,.5)['macro_f1']
    report={'slot':slot,'base':SLOTS[slot]['base'],'method':SLOTS[slot]['method'],
            'validation_status':'COMPLETE_OUTER_3FOLD' if complete_outer else f'PROVISIONAL_{len(thresholds)}_FOLDS',
            'outer_macro_f1':None if outer is None else outer['macro_f1'],'outer_metric':outer,
            'outer_folds':fold_metrics,
            'dev_macro_f1':development['macro_f1'],'dev_metric':development,
            'inner_thresholds':thresholds,'final_threshold':final_threshold,
            'majority_outer_macro_f1':majority_macro,
            'outer_cases':len(validated_rows),'dev_cases':len(dev),'test_cases':len(test),
            'quality_warning':not complete_outer or outer['macro_f1']<majority_macro+.05 or
               min(outer['per_class']['0']['f1'],outer['per_class']['1']['f1'])<.25,
            'private_f1':None,
            'note':'Outer cases never selected their checkpoint or threshold; private score is unknown.'}
    u.write_json(ROOT/'reports'/f'submission_{slot}.json',report)
    return report,test_logits,test


def package_slot(slot):
    report,logits,test=summarize_slot(slot)
    threshold=report['final_threshold']
    predictions=[]
    for row in test:
        probability=float(torch.sigmoid(torch.tensor(logits[row['id']])))
        predictions.append({'id':row['id'],'pred':int(probability>=threshold)})
    if len(predictions)!=len(test) or len({row['id'] for row in predictions})!=len(test):
        raise RuntimeError('Submission contains missing or duplicate IDs')
    if any(type(row['pred']) is not int or row['pred'] not in (0,1) for row in predictions):
        raise RuntimeError('Submission has a nonbinary prediction')
    destination=ROOT/'submissions'/slot
    destination.mkdir(parents=True,exist_ok=True)
    u.write_json(destination/'prediction.json',predictions)
    output=WORK/f'CS781_Phase4_Submission_{slot}.zip'
    temporary=output.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(destination/'prediction.json','prediction.json')
    os.replace(temporary,output)
    with zipfile.ZipFile(output) as archive:
        if archive.namelist()!=['prediction.json'] or json.loads(archive.read('prediction.json'))!=predictions:
            raise RuntimeError('Submission ZIP round-trip verification failed')
    shutil.copy2(output,destination/'submission.zip')
    report['submission']=str(output)
    report['zip_sha256']=u.sha(output)
    report['prediction_sha256']=u.sha(destination/'prediction.json')
    u.write_json(ROOT/'reports'/f'submission_{slot}.json',report)
    log('PACKAGED',slot,'outer macro F1',None if report['outer_macro_f1'] is None else round(report['outer_macro_f1'],4),
        'official dev',round(report['dev_macro_f1'],4),'quality warning',report['quality_warning'])


def final_report():
    summaries=[]
    for slot in SLOTS:
        path=ROOT/'reports'/f'submission_{slot}.json'
        if not path.is_file():raise RuntimeError(f'Missing slot {slot} report')
        summaries.append(u.read_json(path))
    hashes=[row['prediction_sha256'] for row in summaries]
    distinct=len(set(hashes))==len(hashes)
    complete_validation=all(row['validation_status']=='COMPLETE_OUTER_3FOLD' for row in summaries)
    ranked=sorted(summaries,key=lambda row:(-1 if row['outer_macro_f1'] is None else row['outer_macro_f1']),reverse=True)
    result={'status':('SIX_DISTINCT_PACKAGES_READY' if distinct and complete_validation else
                      'SIX_PROVISIONAL_PACKAGES_READY' if distinct else
                      'DUPLICATE_PREDICTIONS_REQUIRE_NEW_EXPERIMENT'),
            'slots':summaries,'distinct_predictions':distinct,'private_scores':[None]*len(summaries),
            'outer_f1_ranking':[{'slot':row['slot'],'outer_macro_f1':row['outer_macro_f1'],
                                 'outer_cases':row['outer_cases'],
                                 'validation_status':row['validation_status']} for row in ranked],
            'artifact':str(ARTIFACT),
            'warning':'Local macro F1 does not guarantee the private Codabench score. '
                      'Each recovery ZIP must be downloaded or persisted outside a fatal draft session.'}
    u.write_json(ROOT/'report.json',result)
    log('FINAL REPORT',result['status'])


def snapshot():
    u.snapshot()


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('action',choices=('prepare','tokenize','train','package','report','snapshot'))
    parser.add_argument('--slot',choices=tuple(SLOTS),default='01')
    parser.add_argument('--base',choices=('01','02','03','04'),default='01')
    parser.add_argument('--stage',default='final')
    parser.add_argument('--chunk-epochs',type=int,default=2)
    args=parser.parse_args()
    if args.action=='prepare':prepare()
    elif args.action=='tokenize':tokenize_base(args.base)
    elif args.action=='train':train_stage(args.slot,args.stage,args.chunk_epochs)
    elif args.action=='package':package_slot(args.slot)
    elif args.action=='report':final_report()
    elif args.action=='snapshot':snapshot()


if __name__=='__main__':main()


In [ ]:
from pathlib import Path

root = Path("/kaggle/input")

print("=== ALL ZIP FILES ===")
for p in root.rglob("*.zip"):
    print(p)

print("\n=== PHASE4 RELATED FILES ===")
for p in root.rglob("*"):
    if p.is_file() and "phase4" in p.name.lower():
        print(p)

## Run all six methods and case-level validation

In [ ]:
from pathlib import Path
import json, subprocess, sys, time

RUNNER=Path('/kaggle/working/cs781_phase4_cjpe_runner.py')
ROOT=Path('/kaggle/working/CS781_Capstone/phase4_cjpe')
ARTIFACT=Path('/kaggle/working/CS781_Phase4_All_Artifacts.zip')
SLOTS=('01','02','03','04','05','06')
BASES={'01':'01','02':'03','03':'04','04':'02','05':'03','06':'03'}
STARTED=time.monotonic()
SESSION_HOURS=9.5
last_archive=0.0

def elapsed_hours():
    return (time.monotonic()-STARTED)/3600

def run(action, *, slot=None, stage=None, base=None, chunk_epochs=None, distributed=False):
    command=[sys.executable]
    if distributed:
        command+=['-m','torch.distributed.run','--standalone','--nproc_per_node=2']
    command+=[str(RUNNER),action]
    if slot is not None:command+=['--slot',slot]
    if stage is not None:command+=['--stage',stage]
    if base is not None:command+=['--base',base]
    if chunk_epochs is not None:command+=['--chunk-epochs',str(chunk_epochs)]
    print('RUN',action,'slot',slot,'stage',stage,'elapsed hours',round(elapsed_hours(),2),flush=True)
    subprocess.run(command,check=True)

def archive(force=False):
    global last_archive
    if force or time.monotonic()-last_archive>=1200 or not ARTIFACT.is_file():
        # No DDP process is alive here. The old ZIP remains intact if this fails.
        print('ARCHIVE after distributed workers exit',flush=True)
        subprocess.run([sys.executable,str(RUNNER),'snapshot'],check=True,timeout=1200)
        last_archive=time.monotonic()

def complete(slot,stage):
    path=ROOT/'runs'/slot/stage/'status.json'
    return path.is_file() and json.loads(path.read_text())['complete']

def ensure_stage(slot,stage):
    while not complete(slot,stage):
        if elapsed_hours()>=SESSION_HOURS:
            print('Session time reserve reached. Recovery ZIP is ready; attach it and Run All in a new session.',flush=True)
            archive(force=True)
            return False
        run('train',slot=slot,stage=stage,chunk_epochs=2,distributed=True)
        archive()
    return True

run('prepare')
archive(force=True)
stopped=False

# Fit all six final models first. This creates individually usable packages
# before the more expensive outer-fold audit. Their reports say PROVISIONAL.
for slot in SLOTS:
    run('tokenize',base=BASES[slot])
    if slot=='06':run('tokenize',base='01')
    if not ensure_stage(slot,'final'):
        stopped=True
        break
    run('package',slot=slot)
    archive(force=True)
    report=json.loads((ROOT/'reports'/f'submission_{slot}.json').read_text())
    print('SLOT',slot,'DEV MACRO F1',round(report['dev_macro_f1'],4),
          'STATUS',report['validation_status'],'WARNING',report['quality_warning'],flush=True)

# Outer folds use their own inner validation for epoch and threshold choices.
# Slot 01 folds precede slot 06 so every distillation fold has a fold-safe teacher.
if not stopped:
    for slot in SLOTS:
        for fold in range(3):
            if not ensure_stage(slot,f'fold_{fold}'):
                stopped=True
                break
            run('package',slot=slot)
            archive(force=True)
            report=json.loads((ROOT/'reports'/f'submission_{slot}.json').read_text())
            print('SLOT',slot,'FOLD',fold,'OUTER MACRO F1',
                  None if report['outer_macro_f1'] is None else round(report['outer_macro_f1'],4),
                  'DEV MACRO F1',round(report['dev_macro_f1'],4),flush=True)
        if stopped:break

if all((ROOT/'reports'/f'submission_{slot}.json').is_file() for slot in SLOTS):
    run('report')
archive(force=True)
print('RUN STATUS:', 'SAFE RESUME REQUIRED' if stopped else 'ALL REQUESTED STAGES COMPLETED',flush=True)
print('Recovery archive:',ARTIFACT,flush=True)


In [ ]:
# ============================================================
# FORCE RESUME ONLY SLOT 02 FROM COMPLETED EPOCH 3
# FAST PATH -- BYPASS SLOT 01 ENTIRELY
# ============================================================

from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import tempfile
import torch

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")

ROOT = WORK / "CS781_Capstone/phase4_cjpe"
RUNNER = WORK / "cs781_phase4_cjpe_runner.py"

SLOT = "02"
STAGE = "final"

stage_dir = ROOT / "runs" / SLOT / STAGE
stage_dir.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("FORCE-RESUME SLOT 02")
print("=" * 70)


# ============================================================
# 1. FIND THE EXTRACTED EPOCH-3 RESUME PATCH
# ============================================================

patches = []

for info_path in INPUT.rglob("resume_info.json"):

    if not info_path.is_file():
        continue

    try:
        info = json.loads(
            info_path.read_text(encoding="utf-8")
        )
    except Exception:
        continue

    if "epoch" not in info:
        continue

    if "relative_stage_directory" not in info:
        continue

    patch_stage = info_path.parent / "stage"

    if not (patch_stage / "latest.pt").is_file():
        continue

    epoch = int(info["epoch"])

    patches.append(
        (
            epoch,
            info_path,
            patch_stage,
            info,
        )
    )


if not patches:
    raise RuntimeError(
        "Could not find an extracted epoch resume patch "
        "under /kaggle/input."
    )


# Highest epoch patch wins.
patches.sort(
    key=lambda x: (
        x[0],
        x[1].stat().st_mtime,
    )
)

saved_epoch, info_path, patch_stage, patch_info = patches[-1]

print("Found resume patch:")
print(info_path.parent)
print("Patch saved epoch:", saved_epoch)


# ============================================================
# 2. REQUIRE EXACTLY EPOCH 3 FOR THIS RESUME
# ============================================================

if saved_epoch != 3:
    raise RuntimeError(
        f"Expected the resume patch to contain epoch 3, "
        f"but it contains epoch {saved_epoch}."
    )


# ============================================================
# 3. OVERLAY PATCH DIRECTLY INTO runs/02/final
# ============================================================

print("\nRestoring epoch-3 Slot-02 state into:")
print(stage_dir)

for filename in (
    "latest.pt",
    "best.pt",
    "history.json",
):

    src = patch_stage / filename

    if not src.is_file():
        print("Patch does not contain:", filename)
        continue

    dst = stage_dir / filename

    with src.open("rb") as reader, tempfile.NamedTemporaryFile(
        dir=stage_dir,
        delete=False
    ) as handle:

        temp = Path(handle.name)

        shutil.copyfileobj(
            reader,
            handle
        )

        handle.flush()
        os.fsync(handle.fileno())

    os.replace(
        temp,
        dst
    )

    print("RESTORED:", dst)


# ============================================================
# 4. VERIFY THE ACTUAL CHECKPOINT CONTENT
# ============================================================

latest = stage_dir / "latest.pt"

if not latest.is_file():
    raise RuntimeError(
        f"Missing checkpoint: {latest}"
    )

try:
    state = torch.load(
        latest,
        map_location="cpu",
        weights_only=False,
    )
except TypeError:
    state = torch.load(
        latest,
        map_location="cpu",
    )

checkpoint_epoch = int(
    state["epoch"]
)

print("\nCheckpoint epoch:", checkpoint_epoch)
print("Best epoch:", state.get("best_epoch"))
print("Best score:", state.get("best"))
print("Best threshold:", state.get("best_threshold"))


if checkpoint_epoch != 3:
    raise RuntimeError(
        f"STOP: latest.pt contains epoch {checkpoint_epoch}, "
        f"not epoch 3."
    )


# ============================================================
# 5. MAKE SURE AN OLD status.json CANNOT CAUSE SLOT 02 TO SKIP
# ============================================================

status_path = stage_dir / "status.json"

if status_path.is_file():

    try:
        status = json.loads(
            status_path.read_text()
        )
    except Exception:
        status = {}

    if status.get("complete"):

        backup = stage_dir / "status.before_force_resume.json"

        shutil.copy2(
            status_path,
            backup
        )

        status_path.unlink()

        print(
            "Removed stale COMPLETE status so Slot 02 can resume."
        )


# ============================================================
# 6. VERIFY TWO GPUS
# ============================================================

gpu_count = torch.cuda.device_count()

print("\nVisible GPUs:", gpu_count)

if gpu_count != 2:
    raise RuntimeError(
        f"This notebook expects exactly 2 GPUs; found {gpu_count}."
    )

for i in range(gpu_count):
    print(
        f"GPU {i}:",
        torch.cuda.get_device_name(i)
    )


# ============================================================
# 7. SAFE PERFORMANCE SETTINGS
#
# These do NOT change the training objective/checkpoint.
# They only avoid unnecessary CPU/thread overhead.
# ============================================================

os.environ["TOKENIZERS_PARALLELISM"] = "true"

# torchrun normally forces OMP_NUM_THREADS=1 if it is unset.
# Give each GPU worker two CPU threads instead.
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"

# Helpful for NCCL failure handling; does not change training.
os.environ["NCCL_ASYNC_ERROR_HANDLING"] = "1"

# Avoid allocator fragmentation.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True"
)


# ============================================================
# 8. MAKE SURE BASE-03 TOKEN CACHE EXISTS
#
# Slot 02 uses base 03.
# This does NOT train Slot 01.
# ============================================================

token_cache = (
    ROOT
    / "hf_cache"
    / "tokens_03.pt"
)

if token_cache.is_file():

    print("\nBase-03 token cache already exists:")
    print(token_cache)

else:

    print(
        "\nBase-03 token cache missing."
    )

    print(
        "Tokenizing Base 03 once before training..."
    )

    subprocess.run(
        [
            sys.executable,
            str(RUNNER),
            "tokenize",
            "--base",
            "03",
        ],
        check=True,
        env=os.environ.copy(),
    )

    if not token_cache.is_file():
        raise RuntimeError(
            "Base-03 tokenization finished but "
            "tokens_03.pt was not created."
        )


# ============================================================
# 9. FINAL PRE-FLIGHT VERIFICATION
# ============================================================

# Reload the checkpoint one final time immediately
# before launching training.
try:
    verify_state = torch.load(
        latest,
        map_location="cpu",
        weights_only=False,
    )
except TypeError:
    verify_state = torch.load(
        latest,
        map_location="cpu",
    )

assert int(verify_state["epoch"]) == 3

print("\n" + "=" * 70)
print("PRE-FLIGHT PASSED")
print("Slot:              02")
print("Stage:             final")
print("Completed epoch:   3")
print("NEXT EPOCH:        4")
print("Slot 01:           WILL NOT BE RUN")
print("=" * 70)


# Free CPU copy before launching workers.
del state
del verify_state


# ============================================================
# 10. LAUNCH ONLY SLOT 02
#
# IMPORTANT OPTIMIZATION:
#
# Your old orchestration used:
#
#     --chunk-epochs 2
#
# which repeatedly killed/restarted torchrun.
#
# We use a large chunk here so the SAME DDP workers continue
# until Slot 02 finishes or early-stops.
#
# latest.pt is STILL saved after EVERY completed epoch.
# ============================================================

command = [
    sys.executable,
    "-m",
    "torch.distributed.run",
    "--standalone",
    "--nproc_per_node=2",
    str(RUNNER),
    "train",
    "--slot",
    "02",
    "--stage",
    "final",
    "--chunk-epochs",
    "99",
]

print("\nLaunching ONLY Slot 02:")
print(" ".join(command))
print()
print("THE FIRST TRAIN LINE MUST SAY:")
print("    TRAIN 02 final epoch 4")
print()

subprocess.run(
    command,
    check=True,
    env=os.environ.copy(),
)


# ============================================================
# 11. SLOT 02 FINISHED
# Package Slot 02 only.
# ============================================================

status_path = stage_dir / "status.json"

if status_path.is_file():

    final_status = json.loads(
        status_path.read_text()
    )

else:

    final_status = {}


if final_status.get("complete"):

    print("\nSlot 02 final training COMPLETE.")

    print("Packaging Slot 02...")

    subprocess.run(
        [
            sys.executable,
            str(RUNNER),
            "package",
            "--slot",
            "02",
        ],
        check=True,
        env=os.environ.copy(),
    )

else:

    print(
        "\nSlot 02 subprocess exited, "
        "but stage is not yet marked complete."
    )


# ============================================================
# 12. CREATE ONE RECOVERY ARCHIVE AT THE END
#
# We deliberately do NOT rebuild the giant ZIP every epoch.
# That was wasting a lot of time.
# ============================================================

print("\nCreating updated recovery archive...")

subprocess.run(
    [
        sys.executable,
        str(RUNNER),
        "snapshot",
    ],
    check=True,
    env=os.environ.copy(),
)

print("\n" + "=" * 70)
print("SLOT 02 RUN FINISHED")
print("=" * 70)

print(
    "Recovery:",
    "/kaggle/working/CS781_Phase4_All_Artifacts.zip"
)

print(
    "Slot-02 submission:",
    "/kaggle/working/CS781_Phase4_Submission_02.zip"
)

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import torch
import json

WORK = Path("/kaggle/working")
ROOT = WORK / "CS781_Capstone/phase4_cjpe"
RUNNER = WORK / "cs781_phase4_cjpe_runner.py"

SLOT = "03"
BASE = "04"
STAGE = "final"

print("=" * 70)
print("SLOT 03 ONLY")
print("=" * 70)

# ------------------------------------------------------------
# 1. Verify GPUs
# ------------------------------------------------------------

gpu_count = torch.cuda.device_count()

print("Visible GPUs:", gpu_count)

if gpu_count != 2:
    raise RuntimeError(
        f"Expected 2 GPUs, found {gpu_count}"
    )

for i in range(gpu_count):
    print(
        f"GPU {i}:",
        torch.cuda.get_device_name(i)
    )


# ------------------------------------------------------------
# 2. Performance settings
# ------------------------------------------------------------

os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["NCCL_ASYNC_ERROR_HANDLING"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True"
)


# ------------------------------------------------------------
# 3. Ensure Base-04 token cache exists
#
# Slot 03 uses Phase-1 Base 04.
# ------------------------------------------------------------

token_cache = (
    ROOT
    / "hf_cache"
    / "tokens_04.pt"
)

if token_cache.is_file():

    print(
        "\nBase-04 token cache already exists:"
    )
    print(token_cache)

else:

    print(
        "\nBase-04 token cache missing."
    )

    print(
        "Tokenizing Base 04..."
    )

    subprocess.run(
        [
            sys.executable,
            str(RUNNER),
            "tokenize",
            "--base",
            "04",
        ],
        check=True,
        env=os.environ.copy(),
    )

    if not token_cache.is_file():
        raise RuntimeError(
            "Tokenization finished but tokens_04.pt "
            "was not created."
        )


# ------------------------------------------------------------
# 4. Inspect existing Slot-03 state
# ------------------------------------------------------------

stage_dir = (
    ROOT
    / "runs"
    / SLOT
    / STAGE
)

latest = stage_dir / "latest.pt"
status_path = stage_dir / "status.json"

if status_path.is_file():

    status = json.loads(
        status_path.read_text()
    )

    if status.get("complete"):
        print(
            "\nSlot 03 final is already complete."
        )

        print(
            "No training needed."
        )

        raise SystemExit


if latest.is_file():

    try:
        state = torch.load(
            latest,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        state = torch.load(
            latest,
            map_location="cpu",
        )

    completed_epoch = int(
        state["epoch"]
    )

    print(
        "\nExisting Slot-03 checkpoint found."
    )

    print(
        "Completed epoch:",
        completed_epoch
    )

    print(
        "Will resume from epoch:",
        completed_epoch + 1
    )

    del state

else:

    print(
        "\nNo Slot-03 checkpoint found."
    )

    print(
        "Slot 03 will start from epoch 1."
    )


# ------------------------------------------------------------
# 5. Launch ONLY Slot 03
#
# Large chunk means torchrun stays alive instead of restarting
# every two epochs.
# ------------------------------------------------------------

command = [
    sys.executable,
    "-m",
    "torch.distributed.run",
    "--standalone",
    "--nproc_per_node=2",
    str(RUNNER),
    "train",
    "--slot",
    SLOT,
    "--stage",
    STAGE,
    "--chunk-epochs",
    "99",
]

print("\nLaunching ONLY Slot 03:")
print(" ".join(command))
print()

subprocess.run(
    command,
    check=True,
    env=os.environ.copy(),
)


# ------------------------------------------------------------
# 6. Package Slot 03 when complete
# ------------------------------------------------------------

status_path = stage_dir / "status.json"

if status_path.is_file():

    status = json.loads(
        status_path.read_text()
    )

else:

    status = {}


if status.get("complete"):

    print(
        "\nSlot 03 final training COMPLETE."
    )

    print(
        "Packaging Slot 03..."
    )

    subprocess.run(
        [
            sys.executable,
            str(RUNNER),
            "package",
            "--slot",
            SLOT,
        ],
        check=True,
        env=os.environ.copy(),
    )

else:

    print(
        "\nTraining process ended, but Slot 03 "
        "is not marked complete yet."
    )


# ------------------------------------------------------------
# 7. Snapshot once at the end
# ------------------------------------------------------------

print(
    "\nCreating updated recovery archive..."
)

subprocess.run(
    [
        sys.executable,
        str(RUNNER),
        "snapshot",
    ],
    check=True,
    env=os.environ.copy(),
)

print("\n" + "=" * 70)
print("SLOT 03 FINISHED")
print("=" * 70)

print(
    "Recovery:",
    "/kaggle/working/CS781_Phase4_All_Artifacts.zip"
)

print(
    "Submission:",
    "/kaggle/working/CS781_Phase4_Submission_03.zip"
)

In [ ]:
from pathlib import Path
import json
import shutil
import zipfile
import torch
import os

ROOT = Path("/kaggle/working/CS781_Capstone/phase4_cjpe")
SLOT = "03"
STAGE = "final"

stage_dir = ROOT / "runs" / SLOT / STAGE
latest = stage_dir / "latest.pt"

if not latest.is_file():
    raise RuntimeError(
        f"No resumable checkpoint found at:\n{latest}\n"
        "This means no Slot-03 epoch completed successfully yet."
    )

# ------------------------------------------------------------
# Verify exactly what epoch has been saved
# ------------------------------------------------------------
try:
    state = torch.load(
        latest,
        map_location="cpu",
        weights_only=False,
    )
except TypeError:
    state = torch.load(
        latest,
        map_location="cpu",
    )

epoch = int(state["epoch"])

print("=" * 70)
print("SLOT 03 RECOVERY")
print("=" * 70)
print("Last COMPLETED epoch:", epoch)
print("Best epoch:", state.get("best_epoch"))
print("Best score:", state.get("best"))
print("Best threshold:", state.get("best_threshold"))

# ------------------------------------------------------------
# Create a SMALL Slot-03-specific resume archive
# ------------------------------------------------------------
patch = Path(
    f"/kaggle/working/"
    f"CS781_Phase4_Slot03_Epoch{epoch}_Resume.zip"
)

files = [
    "latest.pt",
    "best.pt",
    "history.json",
    "status.json",
]

resume_info = {
    "slot": SLOT,
    "stage": STAGE,
    "epoch": epoch,
    "relative_stage_directory": f"runs/{SLOT}/{STAGE}",
    "checkpoint": f"runs/{SLOT}/{STAGE}/latest.pt",
}

with zipfile.ZipFile(
    patch,
    "w",
    compression=zipfile.ZIP_STORED,
    allowZip64=True,
) as z:

    for name in files:
        path = stage_dir / name

        if path.is_file():
            z.write(
                path,
                f"stage/{name}",
            )

            print("Saved:", path)

    z.writestr(
        "resume_info.json",
        json.dumps(
            resume_info,
            indent=2,
        ),
    )

print()
print("Created Slot-03 resume archive:")
print(patch)
print(
    "Size:",
    round(patch.stat().st_size / 2**30, 3),
    "GiB",
)

# ------------------------------------------------------------
# ALSO create/update the full Phase-4 recovery archive
# so Slot 01/02 and every other existing artifact is preserved.
# ------------------------------------------------------------
print()
print("Creating full Phase-4 recovery archive...")

import subprocess, sys

runner = Path("/kaggle/working/cs781_phase4_cjpe_runner.py")

subprocess.run(
    [
        sys.executable,
        str(runner),
        "snapshot",
    ],
    check=True,
)

full_artifact = Path(
    "/kaggle/working/CS781_Phase4_All_Artifacts.zip"
)

if not full_artifact.is_file():
    raise RuntimeError(
        "Full Phase-4 artifact was not created."
    )

print()
print("=" * 70)
print("BACKUP COMPLETE")
print("=" * 70)

print("Slot-03 resume ZIP:")
print(patch)

print()
print("Full Phase-4 recovery ZIP:")
print(full_artifact)

print()
print(
    f"Slot 03 can resume from epoch {epoch + 1} "
    f"using the saved latest.pt."
)

In [ ]:
# ONE-CELL RESUME: Slot 03 (LoRMA) -> Slot 04 (AdaLoRA)
# Run this INSTEAD OF the notebook's old Run-All/orchestration cell.
# On a fresh Kaggle session, attach the latest Phase-4 recovery artifact,
# plus the four Phase-1 datasets; attach a separate Slot-03 patch if you have one.

from pathlib import Path
import os, sys, json, hashlib, base64, zlib, tempfile, shutil, zipfile, subprocess
import torch

WORK=Path('/kaggle/working')
INPUT=Path('/kaggle/input')
ROOT=WORK/'CS781_Capstone/phase4_cjpe'
ROOT.mkdir(parents=True, exist_ok=True)
RUNNER=WORK/'cs781_phase4_cjpe_runner.py'
UTILS=WORK/'cs781_phase4_utils.py'

# Embedded exact code from this notebook: no need to rerun Cells 2/4 after restart.
_EMBEDDED_UTILS='eNrNXG1z28YR/s5fgTgfADo0aKlO22GKziiKXHvqWBpJ6UyraDAgeRARgQCKF0ushv+9u/d+wAEkLTVTfbDJw93e3t7e7rN7C7569epqFZVk6ZRkkX8h5WbiXKyiirw5ctI8WibZ3cSJsqXz6fzy5xNnRdKClJUT5yXr5rzzX716NYrLfO2EYdzUTUnC0EnWRV7WMDDL66hO8qwajXgbjFqlyVx8Zf9Bg9/USSpaf6vyTHxeR/VKfM4r8ala6f2rjXxQk3URJykR3/+TsK+UxQJowVyCvwskLQfm5WLFutGPolOWaY1+lon2uMkWuLQodaLKeT8aXZ6fXwdI0nOn99HdXUqmD3l5DyKcnl796c9H4WlUVHWekWmBonsXLn4riDseffx88Ut7YJIVTQ3PTi6vP74/OR2me8HInaRpeFLWSRwt6sqHdcP4H0+uzsL3Hz+dXQVPsfuUzN4eL7fuLHYpC0dh1czXSVXBMvizUOgBJUA3OnGSzCmj7I54R5Pvx9vR1fkvl6dI0X175M7cJYxYkjJM136xcSfu22NoJBlrbD0cOeLPffsH6FaUJE4ew9VmXiZLReEdUojjZJGQrA4XUVNFKX+6HY1GSxKDAkQebud4RmkukztS1QHXLh+eHn//R29Mnz0k9YruNRvg5wXJPLecu2PcuRWod0pmkjNc8jzNF/e47KQmpZdG6/kymrGOfkmipffuL385fjuezF13rEYqPvymWEY18SgdxkRJ4Gxk4vmKPLJPwCJbD5INUe/1VfFB2Ozjcaw8bRl0RE0ea48KG/QhcJs6/rM7FjQfSuBfEZ18idKGcNJ1VN6ROlD0tFa/AJOQ1f76fpmUHvtSBddlQybkManqML+n32AeKV9x7vzP0ZosrwkekqjcvIcmT0rIfdBUoMW1egCTBgYj2iOSkpoE76O0IrTR2ELZrRbTs/XxncuAs7HqRKW6bNaFZ+wgFdLEaGLjzbYkWwJnwbHZSrIKLWBULZJE45Jyqmbm/NDt8dxfM3fcfhSnTbXyVHNe+XG1yRZiKSjpLPfGnCg8LkmRRgtN1lIGikMm1BHnhh2j6AsJqWn7P1GRr978vfe+Ljdab2rVUQo7tUAStm2pbQd6dqG1ExqhOAFvkm4sa/GbLE2ye5M2tdzZnRC1To3uLZBbhuBvPFz8pCT/bhLw9Gxb2BzgWxcrUgUVODSyVNSfjHkKWFqVpyCisdGOlrJAK0k9mF/epfncY4I2TkoMFJIqxK3XKGxH2sqhDyi0x/kZ//XIAdqKH8E7hSJ6x2+CI0ZGyayMEsAll01WJ2tyVpZ5aUotds8eC7KA9TrkEVxlunFARs4TMr51GjjWpeO2RjzRFW5/gCVDB+eJT7912zvIDTZ/fvP2FlfGvzkE9Nb5DJPJsxeTEKdlQmNLwIPGNFdTWU5WLWSe52lL1CgagFuUAIo7msOeNbUucuzi+r5L+8G+0a5wyurK7PLrr7ILziENxrdO8Iw/GH52cX764c3FyfXpB+fy7PT8H2eX/4Tmq6ZASFU5P55ff5hBg+McjR0D3JwV+WL1OrwkVbMm4QVKFBEK7Xs8dv5OcdEbcIZlRLcW3C+MA5SWp0tErIs8q6Mkg+MCwqQ0kizOffQBzndOVUd3ZPrc9dFNDTn5AlkE6JItE4QClcf3V7UEN7dsd7913sg/518fL5yCK4xqHsnzho9wY/iR1Y+eu4/EAB2oswLKyVVmsVJH1EQ0KLkk08wLteUcVvv/SgpqvykJao8jMKjJF2ISQTWqAv6IGuM0oeCnZSjc9uYYiliZRK3cMd8c54EGmsS8FLp1pxibbBAUXJBktYddblz63b01O5UkhcAGfCdVnaCqS97bfBCCOwNbk5ebNgEuemUDzIHj7lIthi12f8mQAtdppscorSe6H9uZ82SS3bomF0ob/agAWLz0PLrcieOirkyYakyoRAXYMNT1TJ44dgTlwRvQYSSG56NXjy071NFZSWRvvWUnsszzOlCDGcaQfdim0T6q+9SlzcrYpyiwOlCdpy5r8ovabTPKn+zLJdVdFQxITsc6Xa6T4mxgJ3SY/cqn99zBwE71f5bqH6b2QyqvbH1H+aXQhg/AgPID70L56QabJ4C7YzVexnAQ3wP8p1SYB7CY/X4PIbEQikg9UDIpStwX93POdkmsnDkFik18YJsGDxx9q02jLDP00Vo8HkEFLe7JJmARL0a/65mJnrAJYM2k23h864OEwabDf+EaN0wDRxpEQqy9ZCKa3ANCnVR5Uy7IhCq+4urmzdHtyKJuFBrtp2+yOcDUzNTsbvbYN47hG/CewsDuHsxA+nw5nV3gQ6kjdupNQbAzFcCOrjRUYKziEHMZflSFRV4lj97YpCNUCWcIAmrNZ8MunDH+fB+O+Jz2BXSOn8Eb8O/7OrWfmiJNFqAFzpqs5wjfMAOly9l5Yty2/Rm6FzYGh3jMcE+VcZ7wlnnr+yrBk7vh3qbLJqJ4RnYYjvRCEi2elko3pbrM6I59ibT1P7pRQuQ0b8W74y6hlyDlZDCuRu2XE070CNoaPFtDV0sgbevPUrL+Ii82yE0+/83jHLKh9lE9CQ/9rz/5YekponAVeDOxdzvzU3bJjPZSWFEgjseMjeqeTQzilLyqciHcINXHDlRAhaT8gtiYSurKqNRwSAFhkkBONBXUrGiyXPQBjV7N7GildQKqiEjeTJ7+D1XwEPU7XPV2qN1+Kre3uh2uaqMOzsStacPMnRDTal4vddQgDKkDfpcS4yDGUVPZ8mbo6knAsmcYZHms+2QdFWGaL+ilT+AuisadPJDkblVXYZ6lG77v7CQ9LkhRO9fgCilnX0OcswY4sInSUEFXSqGFXUFYer9vAg2JHJI+MsTHMMA6qdbML61JHWEIFDxp1LcTB2KhxX2RA2/Bk85FN4XU1hV2HfeOz7QvzLjUMPEBqEFRUBbMPZW8gwVZF3h+Bfgx1/ID6AhPsjwkaYqz40UZ3p5Zun93tPW1rL9iY6QwI4et+rCXyEP9fPLxs3Px4eTqDAQr7tgOzEh1rtv2zEXpqSgQKbkrk3ojElFlk4XwPE7uaMvLZKRoHliyE65h8jDinNvTUipS5/z1RermAow4nQfa4nkr0OaWy2PRc2vZ7njvcBm2qi5zEGWV3GVgW5x6FdXwTwKYEb8QeX7Eiiljfj8jFZsdw4Fdk3cDSKSyO4zTgzENNKuu478ezbQlXpQkBsWJhAY9rHKwURRSpHl+Xzlpck/oYlHe2VLpnK+FjkgD8/83hcrcqylpWoFdTLs0N0xdLZjfBwKCYHlhIUJLh9tRKwKQ842D4KgV1bPlyx4QTw47rJ+btE7A7mgxv3VXKxYDQ3Sg1iUihE7MjrOqlC2L2+UZkceDB0XiuiRLYuhLcwsS8bXOgNFTy+QYg8c3Lqp35d7a4sreVfYZfnlghZmfoB/CXRKz+hijV143j9bNxByQgxHFAiI4kskWPS6j8JXJSjwe7Y9b7QrBbr6MHJDBCQaI5ULnAmbCWVSKokr+Q76hYAFldePONzXuxz7TX8FQ6fNRygfysYo8aIXwmM3MihXc2z1m/hBVq0Nnbl3/msmQ1sMDr3X5gvhY2qNqbyBfMIfFw2vuXfcZksYdRxVBQxynCbhdFpaIlc/s+meY7tELBTTmFfVeQc2+Ac1hwcxAILM7iBkMYKxo9Ctsk5EabeOObx1O2kFYIolS/O5gxADOjmQOL3ji7eDZ0WpEKcplg46kQqXV3a5HtXw/YAHYLWScB/LKXIBC37w75+GLrvxqtOAL/fqsGz23821qYH/Obe+8m7gYaDkjI12FVwQHJed6jqP7MfsSpYlSAWl68PJSeBzd8siKBeEWey/nOoBSOUrLcpF7QRMWpBbz5qlNabv/8jrL4t4NxQ5hchzTEkiMaobWiu5YZH0GsoWor/A0CNoM27sPJhitbp0aGNx6IWg43Jg/Zw96oO5XAwBaRWETR9sBWdOdz3JEw3lTJob/rYV/TvZ0VxGjNeMKOCR42/u4t5KRiWBHJWNPxowVrWmFjcOLsVRCDq3mO2qUdnTeI2W8R82c5eB0izMBrvAjfoNbd6uAC54oBiHbPTiQ3EM1RFVXq5BrfNhRPM3LsinqA8/iLtg2BOG6chpWna9f8LMBoZDB3qzZu5pZ/4OuHQYTwRYQpWDBpBPi2VmRmCzYkepp43IF5gC5dFGLeeMsWU3TN4JkRT0+nARFSQbkNBT/geUAWUawXuVNbeI8WW3uTmy31v2iH4rZZZPtxoYvSOqP4MREmJx5yqtKMXCkWt0nRUGWLZ45nD1JH6INQC1YVRptaFImIw8wcrpK7lbwv57IdaIYzDKbhlP3ebbCUknwEpnPq88nF1cfzq9fJLNYZVFRrXJ6/inb6Nm5y9bKbjTfrbRYd+IjowY2tRSkFqNO0Smdi6chX5sBttcqX+0Wn/JSR+O7u4rDRbRYEbP6U9RKtgoljZoG5exl3IDqHlZNHCePWv075on9el24+miJilXZrQvqg29luLMjJTOOhGdPrHR2q5VWt2OLoSpwowgfTnP+AKP++E7bCSMcMSXPq7YoJ7ORtSQu8LpWBF+0aTf68k6izmm41jW92j2F8axlVZNYTu6TbFmhSLpMuErwks7AHenIFrjcdKmyTTHab28EO7fBU3cEhxAz+QrLpNuHgYgZrVEzc1ZG363JpkC8DKRZt6E7l8wZWmSxxti6CrFcxbKtPHhmmvfxIry6Pr88+6kPbrC10BMB0Y5RKte29ztn+uns/aeTa8tcvXpiiAZr1syz3wq+TFnIN0Uqrzek7UoPbVh4TzaVWZ1vcnnYCxzCuow694Xadd0vFHEvnSfRe6vVs8euJ9tbijU9fv36D29n/nG8df6W/Dgeupt7AU/Ert+OHHxFTb96ewG3RO/L5+DXPfwnTJbCPy3RyipZodGRX6bKQOIo7UirJ5ycLv6MkCVeqmgSXucQLU6f+GtyN3zQ7VYTKO+zAJwHD+cFMd+Oc9V1cTWtIPRkaA7HyHNzK6+ZPN2amxrq4YqnMtK3eEEjTUHXMupqqe4qzQmQvtEwNc9Vnd+TDJSrnMpP2jUJnaXF2FhzK+KlOZyEFey10nSyp3rR0TTR+o4xqWl7N+w62VxDHrE3M9cPRlT+zWj+Juj00jJzZla3bKGYTdcoWTJAu/NTfRaq7eZ3FEhwI8TyQ+JdYiE7x7X0lbV8fbb7IcowujHnaSkr/n1nfLuxZlIPlIN1E3V6Pg1weiCH5RjYnZ496mwjVH176dwS7LhTS4xrttzaXlvrLI3l1VPPKrt+EQ1YkS4nLQ3ew0ZoOJzpwjO182MmClmkhtoUU2rtTg3V1884bPHFU59dZjoGtGtErarbhcA8nbNP/d9oV8LVAh1tUVx/NNfDJrW4XerdTG23i8zcWisFd75XaqYme98xNbem9b6psTCVBG6HFLtKGYfqYK0mypLCs1wS2u83emSmptw1W9+brQNvt1recNW9rnorfWQ9BRoSK8r8C8mibEHauNyMq8RRpQ7PcnXCzi+7zBvZR4qozCLqSJSBtMdSRDY8skezLIfefvCVq9PLZKRI2xwp4Pi7sXV8O+B5jLd8DTjHoDqVKyLbhr1EuJrwd2EZ3CvIIjB/IsTHthBv36iZk/WgnoHKJqNO6kEkW+hUbaKslZHFCTQUCN+M11c2Fe9d3Yw61vk2YM8U+/SWExwaeQS2+TrlON7Z8gIxp0KFhBTs4Yx44+WFYxqm1rTIyXgxn01mo05H2KiLIGh/UM4hvqkZusQ4a1p0yuvWWIAU6sHuE59qa7aCTQPjVCRLb7w1UnESgITDbOwWRie267ILj16GWW2X2guQD/xTys+PF2fXoot6RvW+AN3D8l0972pf61Cc5/ZkSLF4XBVOqmporcr78FmHw+RJO4fXKU03sYlep97+QQV7wfr/24LaHBvwXtFGffKsnoxpO/ccOxIBit7NAbSUs9BiXvRI6mRb8gA7i/IFjMcj1P35BnGwOk944c5KL6vrSFHW9Ktizv0MgS3qU42hoKsFOwfvm6IWJ9kdKWk20LZzmrTlvF1yA0R+l/2Qy3E0Tvp3xpCTkQYdUlBNSVltWDt73+39BY7ZnGZJ3d6g2iJpzOPIJf2OYqTsNmmExQB9ssPB3L358yZJuZ3x9j3fbdEZThyJUxsY0vvUcJks6r0p0yEa4Yn2YlAJhDp3ht2fR4HpJ12f+HLJa/pzeM9OVi/SqKqcTznQ8rLM/5mCDJWrjp0wTLKkDkMNjpI0nvSss4yye/1mD2BGR+WqpsByfV9S1qI9pO1TtcB/Wu1RABx+AnQQtZSUbnXe1GFMIvz5v2rSSsjoTNEBSVT1/zYWY6JvMgu14emHJ4NZUAz+fZSssR6lyZI4L9ehOSlbv8/wgUk+Ctb0VuvfZe19Px6aAtQvr2yE55xwrzCqRZS2b9/U5tqBBMqpTZD+JFRePkTlsk+fHjv6Qn8HK5CaYXLxaOO5cxglmYFcLZugG5Uy0duj1fd+atEQg79Jf+mcWFFb+sNZ2fFAWvN1d0V06zQh/RcW88Im'
_EMBEDDED_RUNNER='...'
for dst, blob in ((UTILS,_EMBEDDED_UTILS),(RUNNER,_EMBEDDED_RUNNER)):
    if not dst.is_file():
        dst.write_bytes(zlib.decompress(base64.b64decode(blob)))
        print('Recreated helper script:',dst,flush=True)

# Never let an older attached recovery replace more recent local checkpoints.
def copy_atomic(src,dst):
    dst.parent.mkdir(parents=True,exist_ok=True)
    with src.open('rb') as inp, tempfile.NamedTemporaryFile(dir=dst.parent,delete=False) as out:
        tmp=Path(out.name)
        shutil.copyfileobj(inp,out)
    os.replace(tmp,dst)

def ckpt_epoch(path):
    if not path.is_file():return None
    obj=torch.load(path,map_location='cpu',weights_only=False)
    epoch=int(obj['epoch'])
    if not all(k in obj for k in ('model','optimizer','scheduler','scaler','rng')):
        raise RuntimeError(f'Incomplete training state in {path}')
    del obj
    return epoch

# Restore only MISSING files from the main recovered artifact.
# If working already contains run files, keep them; do not erase newer progress.
main_dirs=sorted({p.parent for p in INPUT.rglob('integrity.json')
                  if (p.parent/'run_config.json').is_file() and (p.parent/'runs').is_dir()})
main_zips=sorted(p for p in INPUT.rglob('*.zip') if p.name=='CS781_Phase4_All_Artifacts.zip')
if len(main_dirs)+len(main_zips)>1:
    print('Multiple full artifacts found; using the first folder/ZIP in sorted order.',flush=True)
if main_dirs:
    source=main_dirs[0]
    print('Main Phase-4 recovery folder:',source,flush=True)
    copied=0
    for p in source.rglob('*'):
        if not p.is_file() or p.name=='integrity.json':continue
        rel=p.relative_to(source)
        target=ROOT/rel
        if not target.is_file():
            copy_atomic(p,target);copied+=1
    print('Restored missing main-artifact files:',copied,flush=True)
elif main_zips:
    source=main_zips[0]
    print('Main Phase-4 recovery ZIP:',source,flush=True)
    with zipfile.ZipFile(source) as z:
        for entry in z.infolist():
            p=Path(entry.filename)
            if entry.is_dir() or entry.filename=='integrity.json':continue
            if p.is_absolute() or '..' in p.parts:raise RuntimeError('Unsafe archive member')
            target=ROOT/p
            if target.is_file():continue
            target.parent.mkdir(parents=True,exist_ok=True)
            with z.open(entry) as reader, tempfile.NamedTemporaryFile(dir=target.parent,delete=False) as h:
                temp=Path(h.name);shutil.copyfileobj(reader,h)
            os.replace(temp,target)
else:
    print('No mounted full Phase-4 artifact found. Checking existing /kaggle/working state.',flush=True)

# Restore optional extracted/ZIP resume patches for Slot 03/04, but only if newer.
patches=[]
for info_file in INPUT.rglob('resume_info.json'):
    try:
        info=json.loads(info_file.read_text())
        rel=Path(str(info['relative_stage_directory']))
        if rel.parts not in (('runs','03','final'),('runs','04','final')):continue
        ck=info_file.parent/'stage/latest.pt'
        if ck.is_file():patches.append((int(info['epoch']),rel,ck,info_file.parent,'dir'))
    except Exception:pass
for zp in INPUT.rglob('*.zip'):
    try:
        with zipfile.ZipFile(zp) as z:
            if 'resume_info.json' not in z.namelist() or 'stage/latest.pt' not in z.namelist():continue
            info=json.loads(z.read('resume_info.json'))
            rel=Path(str(info['relative_stage_directory']))
            if rel.parts in (('runs','03','final'),('runs','04','final')):
                patches.append((int(info['epoch']),rel,zp,zp,'zip'))
    except (OSError,zipfile.BadZipFile,ValueError,KeyError):pass
for declared,rel,checkpoint,source,kind in sorted(patches,key=lambda item:item[0]):
    target=ROOT/rel/'latest.pt'
    current=ckpt_epoch(target)
    if current is not None and current>=declared:continue
    if kind=='dir':
        actual=ckpt_epoch(checkpoint)
        if actual!=declared:raise RuntimeError('Resume patch epoch metadata mismatch')
        for name in ('latest.pt','best.pt','history.json'):
            src=source/'stage'/name
            if src.is_file():copy_atomic(src,ROOT/rel/name)
    else:
        with zipfile.ZipFile(source) as z:
            for name in ('latest.pt','best.pt','history.json'):
                member='stage/'+name
                if member not in z.namelist():continue
                dst=ROOT/rel/name
                dst.parent.mkdir(parents=True,exist_ok=True)
                with z.open(member) as reader,tempfile.NamedTemporaryFile(dir=dst.parent,delete=False) as h:
                    tmp=Path(h.name);shutil.copyfileobj(reader,h)
                os.replace(tmp,dst)
        if ckpt_epoch(target)!=declared:raise RuntimeError('ZIP patch checkpoint mismatch')
    print('Applied newer checkpoint:',rel,'completed epoch',declared,flush=True)

# Verify that we did not lose previous Slot-03 training.
slot3_dir=ROOT/'runs/03/final'
slot3_ckpt=slot3_dir/'latest.pt'
slot3_status=slot3_dir/'status.json'
status3=json.loads(slot3_status.read_text()) if slot3_status.is_file() else {}
epoch3=ckpt_epoch(slot3_ckpt)
print('SLOT 03: complete=',status3.get('complete',False),'checkpoint epoch=',epoch3,flush=True)
if epoch3 is None and not status3.get('complete'):
    raise RuntimeError('No Slot-03 latest.pt or completed status. STOP: prior Slot-03 progress is NOT restored. Attach your newest all-artifacts or Slot-03 resume dataset before training.')

if not RUNNER.is_file() or not UTILS.is_file():raise RuntimeError('Missing helper script')
if torch.cuda.device_count()!=2:raise RuntimeError('Enable exactly 2 Kaggle GPUs')

# This keeps the SAME training math/model/schedule. The reliable speedups
# are eliminating repeated DDP launches, completed slots, and tokenization.
env=os.environ.copy()
env['OMP_NUM_THREADS']='2'
env['MKL_NUM_THREADS']='2'
env['TOKENIZERS_PARALLELISM']='true'
env['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'

def call(*args):
    cmd=[sys.executable,str(RUNNER),*map(str,args)]
    print('RUN:', ' '.join(cmd),flush=True)
    subprocess.run(cmd,check=True,env=env)

def ddp(slot):
    cmd=[sys.executable,'-m','torch.distributed.run','--standalone','--nproc_per_node=2',
         str(RUNNER),'train','--slot',slot,'--stage','final','--chunk-epochs','99']
    print('LAUNCH',slot,flush=True)
    subprocess.run(cmd,check=True,env=env)

def have_base(base):
    sources={'02':'encoder_decoder_lm.py','04':'efficient_causal_lm.py'}
    d=ROOT/'bases'/base
    return all((d/p).is_file() for p in (f'model/{sources[base]}','model/course_bpe.py',
            'tokenizer/tokenizer.json','tokenizer/tokenizer_metadata.json','checkpoints/selected_model.pt'))

# Stage official files and Phase-1 bases ONLY if they are missing.
# The original runner's prepare() handles mounted extracted inputs and HF access.
if not (ROOT/'run_config.json').is_file() or any(not (ROOT/'official'/f'{k}.json').is_file() for k in ('train','dev','test')) or not all(have_base(k) for k in ('02','04')):
    call('prepare')

try:
    for slot,base in (('03','04'),('04','02')):
        folder=ROOT/'runs'/slot/'final'
        status_file=folder/'status.json'
        status=json.loads(status_file.read_text()) if status_file.is_file() else {}
        if status.get('complete'):
            print('SKIP training: Slot',slot,'already complete',flush=True)
        else:
            resume=ckpt_epoch(folder/'latest.pt')
            print(f'Slot {slot}: '+(f'resume from epoch {resume+1}' if resume is not None else 'start at epoch 1'),flush=True)
            token=ROOT/'hf_cache'/f'tokens_{base}.pt'
            if not token.is_file():call('tokenize','--base',base)
            ddp(slot)
        status=json.loads(status_file.read_text()) if status_file.is_file() else {}
        if not status.get('complete'):
            raise RuntimeError(f'Slot {slot} exited without a complete status; stopping to protect progress')
        call('package','--slot',slot)
        # Persist both completed slots even if something later fails.
        call('snapshot')
        print('SAVED:',WORK/'CS781_Phase4_All_Artifacts.zip',flush=True)
except KeyboardInterrupt:
    print('Interrupted. Last finished epoch remains in latest.pt if /kaggle/working persists.',flush=True)
    raise
print('FINISHED SLOTS 03 AND 04. Download the latest all-artifacts ZIP!',flush=True)

In [ ]:
# ============================================================
# CS781 PHASE 4 — RESUME SLOT 04 -> SLOT 05 -> SLOT 06
#
# - Never runs Slots 01, 02 or 03
# - Slot 04 must have a checkpoint at epoch >= 6
# - Reuses existing /kaggle/working files
# - Uses both GPUs with DDP
# - Reuses tokenization caches
# - Packages each completed slot
# - Updates the full recovery artifact after each slot
# - Attempts emergency snapshot on interruption/error
# ============================================================

from pathlib import Path
import os
import sys
import json
import time
import zipfile
import shutil
import subprocess
import tempfile
import torch

WORK = Path("/kaggle/working")
ROOT = WORK / "CS781_Capstone" / "phase4_cjpe"

RUNNER = WORK / "cs781_phase4_cjpe_runner.py"
UTILS = WORK / "cs781_phase4_utils.py"

ARTIFACT = WORK / "CS781_Phase4_All_Artifacts.zip"

SLOTS = [
    ("04", "02", "adalora", 12),
    ("05", "03", "adapters", 12),
    ("06", "03", "distill", 10),
]

ROOT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. RESTORE HELPER SCRIPTS IF MISSING
# ------------------------------------------------------------

# Prefer the original scripts already present in working.
# The previous run's snapshot also stores these scripts.

def restore_script(destination, archived_name):

    if destination.is_file():
        print("Existing script:", destination)
        return

    source = ROOT / archived_name

    if not source.is_file():
        raise RuntimeError(
            f"Required script missing:\n{destination}\n"
            f"Also missing: {source}\n"
            "Run notebook Cells 2 and 4 to recreate the "
            "original scripts, then rerun this cell."
        )

    shutil.copy2(source, destination)
    print("Restored script:", destination)


restore_script(RUNNER, "cjpe_runner.py")
restore_script(UTILS, "phase1_utils.py")

# ------------------------------------------------------------
# 2. CHECK GPU AVAILABILITY
# ------------------------------------------------------------

gpu_count = torch.cuda.device_count()

print("\nVisible GPUs:", gpu_count)

if gpu_count != 2:
    raise RuntimeError(
        "Enable Kaggle GPU T4 x2 before training. "
        f"Currently detected: {gpu_count}"
    )

for i in range(gpu_count):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

# ------------------------------------------------------------
# 3. SAFE PERFORMANCE SETTINGS
# ------------------------------------------------------------

ENV = os.environ.copy()

ENV["OMP_NUM_THREADS"] = "2"
ENV["MKL_NUM_THREADS"] = "2"
ENV["TOKENIZERS_PARALLELISM"] = "true"
ENV["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True"
)

# No model architecture, learning rate, batch size,
# optimizer or scheduler modifications.

# ------------------------------------------------------------
# 4. CHECKPOINT INSPECTION
# ------------------------------------------------------------

def checkpoint_epoch(slot):

    checkpoint = (
        ROOT / "runs" / slot / "final" / "latest.pt"
    )

    if not checkpoint.is_file():
        return None

    state = torch.load(
        checkpoint,
        map_location="cpu",
        weights_only=False,
    )

    required = {
        "epoch",
        "model",
        "optimizer",
        "scheduler",
        "scaler",
        "rng",
    }

    missing = required - set(state)

    if missing:
        raise RuntimeError(
            f"Slot {slot} checkpoint is incomplete: "
            f"{sorted(missing)}"
        )

    epoch = int(state["epoch"])
    del state

    return epoch


def slot_status(slot):

    path = ROOT / "runs" / slot / "final" / "status.json"

    if path.is_file():
        return json.loads(path.read_text())

    return {}


# ------------------------------------------------------------
# 5. IMPORTANT: VERIFY SLOT 04 BEFORE TRAINING
# ------------------------------------------------------------

slot04_epoch = checkpoint_epoch("04")
slot04_complete = bool(slot_status("04").get("complete"))

print("\n" + "=" * 70)
print("CHECKING PREVIOUS TRAINING")
print("=" * 70)

print("Slot 04 saved epoch:", slot04_epoch)
print("Slot 04 complete:", slot04_complete)

if not slot04_complete:

    if slot04_epoch is None or slot04_epoch < 6:

        raise RuntimeError(
            "STOPPED TO PROTECT YOUR PROGRESS.\n"
            "Slot 04 epoch-6 checkpoint was not found "
            "in the working directory.\n"
            "Restore the newest Slot-04 recovery checkpoint "
            "before training. No slot was restarted."
        )

    print(
        f"Slot 04 will resume from epoch "
        f"{slot04_epoch + 1}"
    )

else:
    print("Slot 04 already complete; will skip training.")

# ------------------------------------------------------------
# 6. RUNNER HELPERS
# ------------------------------------------------------------

def run_command(*arguments):

    command = [
        sys.executable,
        str(RUNNER),
        *map(str, arguments),
    ]

    print("\nRUN:", " ".join(command), flush=True)

    subprocess.run(
        command,
        check=True,
        env=ENV,
    )


def train_slot(slot):

    command = [
        sys.executable,
        "-m",
        "torch.distributed.run",
        "--standalone",
        "--nproc_per_node=2",
        str(RUNNER),
        "train",
        "--slot",
        slot,
        "--stage",
        "final",
        "--chunk-epochs",
        "99",
    ]

    print("\n" + "=" * 70)
    print("TRAINING SLOT", slot)
    print("=" * 70)

    print(" ".join(command), flush=True)

    subprocess.run(
        command,
        check=True,
        env=ENV,
    )


# ------------------------------------------------------------
# 7. SNAPSHOT + VERIFY
# ------------------------------------------------------------

def save_snapshot():

    print("\nCreating full Phase-4 recovery ZIP...")

    run_command("snapshot")

    if not ARTIFACT.is_file():
        raise RuntimeError(
            "Full Phase-4 recovery archive missing."
        )

    with zipfile.ZipFile(ARTIFACT) as archive:

        names = set(archive.namelist())

        required = {
            "run_config.json",
            "integrity.json",
        }

        if not required.issubset(names):
            raise RuntimeError(
                "Recovery ZIP missing required metadata."
            )

        # Check that all currently available slot
        # checkpoints are present in the ZIP.
        for slot in ("01", "02", "03", "04", "05", "06"):

            local = (
                ROOT / "runs" / slot / "final" / "latest.pt"
            )

            member = f"runs/{slot}/final/latest.pt"

            if local.is_file() and member not in names:
                raise RuntimeError(
                    f"Recovery ZIP missing {member}"
                )

    print(
        "VERIFIED FULL RECOVERY ARCHIVE:",
        ARTIFACT,
        flush=True,
    )

    print(
        "Size:",
        round(ARTIFACT.stat().st_size / 2**30, 3),
        "GiB",
        flush=True,
    )


# ------------------------------------------------------------
# 8. OPTIONAL: SAVE A SMALLER RESUME ZIP PER ACTIVE SLOT
# ------------------------------------------------------------

def save_slot_resume(slot):

    epoch = checkpoint_epoch(slot)

    if epoch is None:
        return

    stage = ROOT / "runs" / slot / "final"

    output = WORK / (
        f"CS781_Phase4_Slot{slot}_"
        f"Epoch{epoch}_Resume.zip"
    )

    temporary = output.with_suffix(".zip.tmp")

    info = {
        "slot": slot,
        "stage": "final",
        "epoch": epoch,
        "relative_stage_directory": (
            f"runs/{slot}/final"
        ),
    }

    with zipfile.ZipFile(
        temporary,
        "w",
        allowZip64=True,
    ) as archive:

        for filename in (
            "latest.pt",
            "best.pt",
            "history.json",
            "status.json",
        ):

            source = stage / filename

            if source.is_file():

                archive.write(
                    source,
                    f"stage/{filename}",
                    compress_type=zipfile.ZIP_STORED,
                )

        archive.writestr(
            "resume_info.json",
            json.dumps(info, indent=2),
        )

    os.replace(temporary, output)

    print(
        f"Slot {slot} epoch {epoch} resume ZIP:",
        output,
        flush=True,
    )


# ------------------------------------------------------------
# 9. PREPARE MISSING TOKEN CACHES ONLY
# ------------------------------------------------------------

def ensure_token_cache(base):

    path = ROOT / "hf_cache" / f"tokens_{base}.pt"

    if path.is_file():
        print("Reusing token cache:", path)
        return

    print("Tokenizing Base", base)

    run_command(
        "tokenize",
        "--base",
        base,
    )

    if not path.is_file():
        raise RuntimeError(
            f"Base-{base} tokenization failed."
        )


# ------------------------------------------------------------
# 10. TRAIN SLOT 04 -> SLOT 05 -> SLOT 06
# ------------------------------------------------------------

current_slot = None

try:

    for slot, base, method, max_epochs in SLOTS:

        current_slot = slot

        print("\n" + "=" * 70)
        print(
            f"SLOT {slot} | {method.upper()} | "
            f"BASE {base} | MAX {max_epochs} EPOCHS"
        )
        print("=" * 70)

        status = slot_status(slot)
        epoch = checkpoint_epoch(slot)

        if status.get("complete"):

            print(
                f"Slot {slot} already completed. "
                "Skipping training."
            )

        else:

            if epoch is None:

                if slot == "04":
                    raise RuntimeError(
                        "Slot 04 resume checkpoint missing."
                    )

                print(
                    f"Slot {slot}: starting from epoch 1."
                )

            else:

                print(
                    f"Slot {slot}: completed epoch {epoch}"
                )

                print(
                    f"Resuming from epoch {epoch + 1}"
                )

            ensure_token_cache(base)

            train_slot(slot)

        # ----------------------------------------------------
        # 11. VERIFY COMPLETION BEFORE PACKAGING
        # ----------------------------------------------------

        status = slot_status(slot)

        if not status.get("complete"):

            raise RuntimeError(
                f"Slot {slot} training ended without "
                "a completed status. "
                "Stopping before the next slot."
            )

        print(
            f"\nSLOT {slot} TRAINING COMPLETE."
        )

        # ----------------------------------------------------
        # 12. GENERATE SUBMISSION AND SAVE RECOVERY
        # ----------------------------------------------------

        run_command(
            "package",
            "--slot",
            slot,
        )

        save_slot_resume(slot)

        save_snapshot()

        print(
            f"\nSLOT {slot} COMPLETED, "
            "PACKAGED AND BACKED UP.",
            flush=True,
        )

except BaseException as exc:

    print(
        "\nTRAINING STOPPED:",
        repr(exc),
        flush=True,
    )

    # Preserve last completed epoch even if interrupted.
    if current_slot is not None:

        try:
            save_slot_resume(current_slot)
        except Exception as backup_error:
            print(
                "Slot backup failed:",
                repr(backup_error),
                flush=True,
            )

    try:
        save_snapshot()
    except Exception as backup_error:
        print(
            "Full snapshot failed:",
            repr(backup_error),
            flush=True,
        )

    raise

# ------------------------------------------------------------
# 13. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ALL REQUESTED SLOTS FINISHED")
print("=" * 70)

for slot, _, method, _ in SLOTS:

    print(
        f"Slot {slot} ({method}):",
        "complete =",
        bool(slot_status(slot).get("complete")),
        "saved epoch =",
        checkpoint_epoch(slot),
    )

print("\nFull recovery archive:")
print(ARTIFACT)

print("\nSubmission ZIPs:")

for slot, _, _, _ in SLOTS:

    submission = (
        WORK / f"CS781_Phase4_Submission_{slot}.zip"
    )

    print(
        f"Slot {slot}:",
        submission,
        "EXISTS" if submission.is_file() else "MISSING",
    )

print(
    "\nIMPORTANT: Download the updated recovery "
    "archive before ending the Kaggle session."
)

In [ ]:
# ==============================================================
# CS781 PHASE 4 — FAST SLOT 06 ONLY
#
# 5 epochs maximum
# 20% training cases per epoch
# 3 document chunks instead of 8
# 384 tokens per chunk instead of 640
#
# Uses 2 GPUs
# Preserves existing Slots 01-05
# Packages Submission 06
# Rebuilds complete Phase-4 recovery archive
# ==============================================================

from pathlib import Path
import os, sys, re, json, shutil, subprocess, zipfile, time
import torch

W = Path("/kaggle/working")
ROOT = W / "CS781_Capstone" / "phase4_cjpe"

ORIGINAL = W / "cs781_phase4_cjpe_runner.py"
UTILS = W / "cs781_phase4_utils.py"
FAST = W / "cs781_phase4_slot06_fast_runner.py"

STAGE = ROOT / "runs" / "06" / "final"
ARTIFACT = W / "CS781_Phase4_All_Artifacts.zip"
SUBMISSION = W / "CS781_Phase4_Submission_06.zip"

STAGE.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 1. Find original scripts in existing working files
# --------------------------------------------------------------

if not ORIGINAL.is_file():
    source = ROOT / "cjpe_runner.py"
    if source.is_file():
        shutil.copy2(source, ORIGINAL)
    else:
        raise FileNotFoundError(
            "Original Phase-4 runner not found. "
            "Run notebook Cell 4 first."
        )

if not UTILS.is_file():
    source = ROOT / "phase1_utils.py"
    if source.is_file():
        shutil.copy2(source, UTILS)
    else:
        raise FileNotFoundError(
            "Phase-4 utilities missing. "
            "Run notebook Cell 2 first."
        )

if not (ROOT / "run_config.json").is_file():
    raise RuntimeError(
        "Existing Phase-4 state missing. "
        "Restore the previous working files first."
    )

# --------------------------------------------------------------
# 2. Check two GPUs
# --------------------------------------------------------------

ngpu = torch.cuda.device_count()

print("GPUs available:", ngpu)

if ngpu != 2:
    raise RuntimeError(
        "Select Kaggle GPU T4 x2. "
        "This runner requires exactly 2 GPUs."
    )

for i in range(ngpu):
    print("GPU", i, torch.cuda.get_device_name(i))

# --------------------------------------------------------------
# 3. Create an isolated, faster Slot-06 runner
#
# IMPORTANT: Do not overwrite the original runner.
# This preserves the existing Slots 01-05 implementation.
# --------------------------------------------------------------

source = ORIGINAL.read_text(encoding="utf-8")

def replace_exact(old, new, label):
    global source

    count = source.count(old)

    if count != 1:
        raise RuntimeError(
            f"Cannot safely change {label}: "
            f"expected 1 match, found {count}"
        )

    source = source.replace(old, new, 1)

# Limit final Slot-06 training to five epochs.
replace_exact(
    "'06': {'base':'03','method':'distill',"
    "'epochs':10,'min_epochs':4",
    "'06': {'base':'03','method':'distill',"
    "'epochs':5,'min_epochs':2",
    "Slot-06 epoch settings"
)

# Reduce the number of chunks processed per document.
replace_exact(
    "MAX_CHUNKS = 8",
    "MAX_CHUNKS = 3",
    "maximum document chunks"
)

# Shorter chunks reduce transformer attention computation.
replace_exact(
    "MAX_TOKENS = 640",
    "MAX_TOKENS = 384",
    "tokens per document chunk"
)

# Train on approximately 20% of cases each epoch.
# The subset changes deterministically with the epoch.
# It preserves two-GPU and gradient-accumulation alignment.
replace_exact(
    "    original=list(ids)\n"
    "    while len(ids)%(2*ACCUM):",
    "    if slot == '06' and stage == 'final':\n"
    "        ids = ids[:max(2*ACCUM, len(ids)//5)]\n"
    "    original=list(ids)\n"
    "    while len(ids)%(2*ACCUM):",
    "20-percent epoch sampling"
)

# Validate Python syntax before launching training.
compile(source, str(FAST), "exec")
FAST.write_text(source, encoding="utf-8")

# Save modified runner and settings INSIDE artifact directory.
shutil.copy2(FAST, ROOT / "slot06_fast_runner.py")

fast_config = {
    "slot": "06",
    "base": "03",
    "method": "distill",
    "maximum_epochs": 5,
    "minimum_epochs": 2,
    "training_fraction_per_epoch": 0.20,
    "max_tokens": 384,
    "max_chunks": 3,
    "gpus": 2,
    "note": (
        "Reduced-budget experiment. Not directly equivalent "
        "to the original Slot-06 configuration."
    ),
}

(ROOT / "slot06_fast_config.json").write_text(
    json.dumps(fast_config, indent=2)
)

print("\nFAST SLOT 06 CONFIGURED")
print(json.dumps(fast_config, indent=2))

# --------------------------------------------------------------
# 4. Check existing Slot-06 state
# --------------------------------------------------------------

def get_status():
    path = STAGE / "status.json"
    return json.loads(path.read_text()) if path.is_file() else {}

def get_epoch():
    path = STAGE / "latest.pt"

    if not path.is_file():
        return None

    state = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )
    epoch = int(state["epoch"])
    del state
    return epoch

saved_epoch = get_epoch()
status = get_status()

print("\nSlot 06 checkpoint epoch:", saved_epoch)
print("Slot 06 completed:", bool(status.get("complete")))

if saved_epoch is not None:
    print("Existing checkpoint detected. Will resume.")

# --------------------------------------------------------------
# 5. Safe environment settings
# --------------------------------------------------------------

ENV = os.environ.copy()
ENV["OMP_NUM_THREADS"] = "2"
ENV["MKL_NUM_THREADS"] = "2"
ENV["TOKENIZERS_PARALLELISM"] = "true"
ENV["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True"
)

def run(command):
    print("\nRUNNING:", " ".join(map(str, command)), flush=True)
    subprocess.run(
        list(map(str, command)),
        check=True,
        env=ENV,
    )

def runner(action, *args):
    run([
        sys.executable,
        str(FAST),
        action,
        *args
    ])

# --------------------------------------------------------------
# 6. Back up latest completed Slot-06 checkpoint separately
# --------------------------------------------------------------

def backup_slot06():

    if not (STAGE / "latest.pt").is_file():
        return

    epoch = get_epoch()

    path = W / (
        f"CS781_Phase4_Slot06_Epoch{epoch}_Resume.zip"
    )

    temporary = W / (path.name + ".tmp")

    info = {
        "slot": "06",
        "stage": "final",
        "epoch": epoch,
        "relative_stage_directory": "runs/06/final",
        "fast_config": fast_config,
    }

    with zipfile.ZipFile(
        temporary,
        "w",
        allowZip64=True
    ) as z:

        for name in (
            "latest.pt",
            "best.pt",
            "history.json",
            "status.json",
            "teacher_logits.pt",
        ):

            file = STAGE / name

            if file.is_file():
                z.write(
                    file,
                    f"stage/{name}",
                    compress_type=zipfile.ZIP_STORED
                )

        z.writestr(
            "resume_info.json",
            json.dumps(info, indent=2)
        )

    os.replace(temporary, path)
    print("\nSlot-06 resume ZIP:", path, flush=True)

# --------------------------------------------------------------
# 7. Full artifact snapshot
# --------------------------------------------------------------

def snapshot():

    print("\nUpdating COMPLETE Phase-4 artifact...")

    runner("snapshot")

    if not ARTIFACT.is_file():
        raise RuntimeError("Full recovery ZIP not created.")

    with zipfile.ZipFile(ARTIFACT) as z:
        names = set(z.namelist())

        for required in (
            "run_config.json",
            "slot06_fast_runner.py",
            "slot06_fast_config.json",
        ):
            if required not in names:
                raise RuntimeError(
                    f"Recovery ZIP missing {required}"
                )

        for slot in ("01", "02", "03", "04", "05", "06"):
            member = (
                f"runs/{slot}/final/status.json"
            )
            local = ROOT / member

            if local.is_file() and member not in names:
                raise RuntimeError(
                    f"Recovery ZIP missing {member}"
                )

    print("FULL ARTIFACT CREATED:", ARTIFACT)
    print(
        "Size:",
        round(ARTIFACT.stat().st_size / 2**30, 3),
        "GiB"
    )

# --------------------------------------------------------------
# 8. Train only Slot 06
# --------------------------------------------------------------

try:

    if not get_status().get("complete"):

        # Reuse existing Base-03 tokenization.
        token_cache = ROOT / "hf_cache" / "tokens_03.pt"

        if not token_cache.is_file():

            print("\nCreating Base-03 token cache...")
            runner("tokenize", "--base", "03")

        if not token_cache.is_file():
            raise RuntimeError("Base-03 token cache missing.")

        # If teacher logits are not yet cached, the original
        # distillation teacher must run once. Its setup can take
        # substantial time and needs the Base-01 token cache.
        teacher_cache = STAGE / "teacher_logits.pt"

        if not teacher_cache.is_file():

            teacher_tokens = (
                ROOT / "hf_cache" / "tokens_01.pt"
            )

            if not teacher_tokens.is_file():
                print("\nCreating teacher Base-01 tokens...")
                runner("tokenize", "--base", "01")

            if not (
                ROOT / "runs" / "01" / "final" / "best.pt"
            ).is_file():
                raise RuntimeError(
                    "Slot-01 teacher best.pt missing. "
                    "Cannot perform genuine Slot-06 distillation."
                )

            print(
                "\nTeacher logits are not cached. "
                "Teacher inference will run before training."
            )

        # Start or resume Slot 06.
        # Only five epochs maximum.
        run([
            sys.executable,
            "-m",
            "torch.distributed.run",
            "--standalone",
            "--nproc_per_node=2",
            str(FAST),
            "train",
            "--slot", "06",
            "--stage", "final",
            "--chunk-epochs", "99",
        ])

    # ----------------------------------------------------------
    # 9. Check completion
    # ----------------------------------------------------------

    status = get_status()

    if not status.get("complete"):

        backup_slot06()

        raise RuntimeError(
            "Slot 06 is not marked complete. "
            "Latest completed epoch was backed up."
        )

    print("\nSLOT 06 TRAINING COMPLETE")
    print("Final epoch:", status.get("epoch"))
    print("Best epoch:", status.get("best_epoch"))

    # ----------------------------------------------------------
    # 10. Produce official Slot-06 submission
    # ----------------------------------------------------------

    runner("package", "--slot", "06")

    if not SUBMISSION.is_file():
        raise RuntimeError(
            "Slot-06 submission ZIP was not created."
        )

    with zipfile.ZipFile(SUBMISSION) as z:

        if z.namelist() != ["prediction.json"]:
            raise RuntimeError(
                "Unexpected submission ZIP contents."
            )

        predictions = json.loads(
            z.read("prediction.json")
        )

        print(
            "\nSubmission predictions:",
            len(predictions)
        )

    # ----------------------------------------------------------
    # 11. Attempt complete six-submission report
    # ----------------------------------------------------------

    all_reports_exist = all(
        (ROOT / "reports" /
         f"submission_{slot:02d}.json").is_file()
        for slot in range(1, 7)
    )

    if all_reports_exist:

        try:
            runner("report")

        except Exception as exc:
            print(
                "Final report warning:",
                repr(exc)
            )

    # ----------------------------------------------------------
    # 12. Final artifact
    # ----------------------------------------------------------

    snapshot()

    print("\n" + "=" * 65)
    print("SLOT 06 AND PHASE-4 ARTIFACTS COMPLETED")
    print("=" * 65)

    print("\nSubmission 06:")
    print(SUBMISSION)

    print("\nFull Phase-4 recovery:")
    print(ARTIFACT)

    print("\nSlot-06 training configuration:")
    print(ROOT / "slot06_fast_config.json")

except BaseException:

    print(
        "\nTraining interrupted or failed. "
        "Saving available completed-epoch progress...",
        flush=True
    )

    try:
        backup_slot06()
    except Exception as e:
        print("Checkpoint backup failed:", repr(e))

    try:
        snapshot()
    except Exception as e:
        print("Full artifact backup failed:", repr(e))

    raise

In [ ]:
# ============================================================
# EMERGENCY BACKUP — SAVE ALL COMPLETED PROGRESS
# SLOT 03 + SLOT 04 + FULL PHASE-4 ARTIFACTS
#
# 1. Interrupt the currently running training cell.
# 2. Run THIS cell.
# 3. Download the ZIP files before the session ends.
# ============================================================

from pathlib import Path
import os
import sys
import json
import zipfile
import subprocess
import torch
import time

WORK = Path("/kaggle/working")
ROOT = WORK / "CS781_Capstone/phase4_cjpe"
RUNNER = WORK / "cs781_phase4_cjpe_runner.py"

print("=" * 70)
print("EMERGENCY PHASE-4 CHECKPOINT BACKUP")
print("=" * 70, flush=True)

if not ROOT.is_dir():
    raise RuntimeError("Phase-4 working directory not found.")

backup_files = []

# ------------------------------------------------------------
# 1. BACK UP SLOT 03 AND SLOT 04 FIRST
# This is prioritized over the larger full archive.
# ------------------------------------------------------------

for slot in ("03", "04"):

    stage = ROOT / "runs" / slot / "final"
    latest = stage / "latest.pt"

    if not latest.is_file():
        print(f"SLOT {slot}: No latest.pt found.")
        continue

    # Verify saved checkpoint
    state = torch.load(
        latest,
        map_location="cpu",
        weights_only=False
    )

    epoch = int(state["epoch"])

    print(
        f"\nSLOT {slot}: "
        f"Last completed epoch = {epoch}",
        flush=True
    )

    del state

    output = WORK / (
        f"CS781_Phase4_Slot{slot}_"
        f"Epoch{epoch}_Resume.zip"
    )

    temporary = output.with_suffix(".zip.tmp")

    metadata = {
        "slot": slot,
        "stage": "final",
        "epoch": epoch,
        "relative_stage_directory": f"runs/{slot}/final",
        "created_at": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    }

    with zipfile.ZipFile(
        temporary,
        "w",
        allowZip64=True
    ) as archive:

        for filename in (
            "latest.pt",
            "best.pt",
            "history.json",
            "status.json"
        ):

            source = stage / filename

            if source.is_file():

                archive.write(
                    source,
                    f"stage/{filename}",
                    compress_type=zipfile.ZIP_STORED
                )

                print(
                    "  Saved:",
                    filename,
                    flush=True
                )

        archive.writestr(
            "resume_info.json",
            json.dumps(metadata, indent=2)
        )

    os.replace(temporary, output)

    # Verify ZIP integrity
    with zipfile.ZipFile(output) as archive:
        bad = archive.testzip()
        if bad is not None:
            raise RuntimeError(
                f"Corrupt file in backup: {bad}"
            )

    backup_files.append(output)

    print(
        "BACKUP CREATED:",
        output,
        flush=True
    )

    print(
        "Size:",
        round(output.stat().st_size / 2**30, 3),
        "GiB",
        flush=True
    )

# ------------------------------------------------------------
# 2. SAVE COMPLETE PHASE-4 WORKING STATE
# Includes all previously restored/trained artifacts,
# excluding regenerable hf_cache, as per notebook snapshot.
# ------------------------------------------------------------

print("\nCreating full Phase-4 recovery archive...")
print("This may take several minutes.", flush=True)

if RUNNER.is_file():

    result = subprocess.run(
        [
            sys.executable,
            str(RUNNER),
            "snapshot"
        ],
        check=False
    )

    if result.returncode != 0:
        print(
            "WARNING: Full snapshot failed. "
            "Slot-specific backups remain available.",
            flush=True
        )

else:
    print(
        "WARNING: Runner missing; "
        "skipping full snapshot.",
        flush=True
    )

full_archive = (
    WORK / "CS781_Phase4_All_Artifacts.zip"
)

if full_archive.is_file():

    # Validate that ZIP can be opened and has entries
    try:
        with zipfile.ZipFile(full_archive) as z:
            if not z.namelist():
                raise RuntimeError("Empty ZIP")

        backup_files.append(full_archive)

        print(
            "Full recovery archive:",
            full_archive,
            flush=True
        )

    except Exception as exc:
        print(
            "WARNING: Full archive validation failed:",
            exc,
            flush=True
        )

# ------------------------------------------------------------
# 3. FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BACKUP PROCESS FINISHED")
print("=" * 70)

for file in backup_files:
    print(
        file.name,
        "|",
        round(file.stat().st_size / 2**30, 3),
        "GiB",
        flush=True
    )

print(
    "\nIMPORTANT: Download these files "
    "from Kaggle /kaggle/working BEFORE "
    "ending the session.",
    flush=True
)

print(
    "On the next session, attach the newest "
    "recovery files and resume from the "
    "last saved completed epoch.",
    flush=True
)

## Verify the ZIPs and show download links

In [ ]:
from pathlib import Path
import hashlib,json,zipfile
from IPython.display import display,FileLink

root=Path('/kaggle/working/CS781_Capstone/phase4_cjpe')
artifact=Path('/kaggle/working/CS781_Phase4_All_Artifacts.zip')
test=json.loads((root/'official/test.json').read_text())
if isinstance(test,dict):
    test=test.get('data',test.get('examples',test.get('cases',test)))
    if isinstance(test,dict):expected=set(test)
    else:expected={row.get('id',row.get('case_id')) for row in test}
else:expected={row.get('id',row.get('case_id')) for row in test}
packages=sorted(Path('/kaggle/working').glob('CS781_Phase4_Submission_*.zip'))
for path in packages:
    with zipfile.ZipFile(path) as z:
        assert z.namelist()==['prediction.json']
        rows=json.loads(z.read('prediction.json'))
    assert isinstance(rows,list) and {row['id'] for row in rows}==expected
    assert len(rows)==len(expected) and all(type(row['pred']) is int and row['pred'] in (0,1) for row in rows)
    print('VERIFIED',path.name,len(rows),'cases')
    display(FileLink(str(path)))
with zipfile.ZipFile(artifact) as z:
    manifest=json.loads(z.read('integrity.json'))['files']
    assert set(manifest)==set(z.namelist())-{'integrity.json'}
    for name,metadata in manifest.items():
        digest=hashlib.sha256();size=0
        with z.open(name) as member:
            for block in iter(lambda:member.read(4<<20),b''):
                digest.update(block);size+=len(block)
        assert digest.hexdigest()==metadata['sha256'] and size==metadata['bytes'],name
print('VERIFIED RECOVERY',artifact.name,'packages',len(packages))
display(FileLink(str(artifact)))
if (root/'report.json').is_file():
    summary=json.loads((root/'report.json').read_text())
    print('FINAL STATUS',summary['status'])
    for row in summary['slots']:
        print(row['slot'],row['method'],'outer F1',row['outer_macro_f1'],
              'dev F1',row['dev_macro_f1'],'warning',row['quality_warning'])
